# Baseline LoRA · función de la cita (rama LLM)

Entrena un clasificador de las 9 funciones de cita sobre un LLM decoder con un adaptador LoRA intercambiable, elige el mejor punto del entrenamiento por macro-F1 de dev (early stopping) y lo evalúa.

- La celda 1 importa todas las librerías, revisa las versiones y prepara el token de Hugging Face. Las celdas 2 a 12 solo definen funciones; la 13 las ejecuta en orden con `ejecutar_experimento()`.
- El modelo se elige con `MODELO` (un id de Hugging Face) y el adaptador con `LORA_TIPO` (la celda 8 lista las variantes).
- Durante el entrenamiento se evalúa dev `EVALS_POR_EPOCA` veces por época. Si el macro-F1 no mejora en `PACIENCIA` evaluaciones seguidas, el entrenamiento para. Al final se restauran los pesos del mejor punto.
- Con `MLFLOW_IP` (y `MLFLOW_EXPERIMENTO`) la corrida se registra también en MLflow mientras entrena, con el nombre `NOMBRE_CORRIDA`. Si el servidor se cae, el entrenamiento sigue y se reintenta en segundo plano; los resultados locales se guardan igual (celda 11).
- bf16, TF32, `torch.compile`, AdamW fusionado y QLoRA se activan solos si el hardware los soporta. Si no, se desactivan y la celda 3 dice por qué.
- Cada campo de `Config` acepta una variable de entorno con su nombre en mayúsculas: `LORA_TIPO=dora`, `LR=1e-4`, `CONFIGURACION=B`. Con la CLI de Colab: `colab exec ... --env LORA_TIPO=dora`.
- `MODO=mini` hace una prueba corta (subconjunto y pocos pasos) para verificar el pipeline en local; `MODO=completo` usa todos los datos. Con `MODO=auto`, el valor por defecto, se elige `completo` en Colab con GPU y `mini` en cualquier otro lugar.
- `MODELO=humo` usa un modelo diminuto con pesos aleatorios y no descarga nada: sirve para depurar sin red; sus métricas no significan nada.
- Token de Hugging Face (`HF_TOKEN`), solo necesario para modelos con acceso restringido: en local se lee de la variable de entorno; en Colab, del secret `HF_TOKEN`. La CLI de Colab no puede leer secrets, así que con ella se pasa como variable: `--env HF_TOKEN=$HF_TOKEN`.

In [ ]:
# @title 1 · Librerías y entorno
# Todos los imports del notebook están en esta celda. La única excepción es crear_modelo (celda 7), que importa
# las clases propias de una familia de modelos solo cuando se usan.
import os
import sys
import warnings

# Variables que deben fijarse antes de importar torch y transformers.
# expandable_segments reduce la fragmentación de la memoria de CUDA. Se usa el nombre clásico de la variable
# porque lo leen todas las versiones de torch (PYTORCH_ALLOC_CONF solo existe desde la 2.9).
if sys.platform == "linux" and not any(v in os.environ for v in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_ALLOC_CONF")):
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")  # las barras con \r ensucian la salida de la CLI
# Cliente de MLflow (solo con MLFLOW_IP): pocos reintentos y espera corta por pedido. Si el servidor cae, quien
# reintenta es RegistroMLflow (celda 11), en segundo plano y sin frenar el entrenamiento.
os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "2")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_BACKOFF_FACTOR", "1")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "30")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")  # mlflow 3 manda estadísticas de uso a sus servidores
# Jupyter local sin ipywidgets: tqdm avisa al importarse aunque el notebook no muestra barras.
warnings.filterwarnings("ignore", message="IProgress not found")

import contextlib
import dataclasses
import functools
import gc
import itertools
import json
import math
import platform
import queue
import random
import shutil
import threading
import time
import urllib.request
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass, fields
from datetime import datetime, timedelta
from importlib import metadata
from pathlib import Path

import numpy as np
import peft
import torch
import torch.nn.functional as F
import transformers
from huggingface_hub import constants as hf_constants
from packaging.version import Version
from peft import (EvaConfig, LoraConfig, TaskType, VeraConfig, get_peft_model, initialize_lora_eva_weights,
                  prepare_model_for_kbit_training)
from peft.optimizers import create_lorafa_optimizer, create_loraplus_optimizer
from peft.utils import get_peft_model_state_dict
from safetensors.torch import load_file, save_file
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from tokenizers import Tokenizer, decoders, pre_tokenizers, trainers
from tokenizers.models import BPE
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LRScheduler
from tqdm import tqdm  # barra de texto: se ve en Jupyter, en Colab y en la terminal de la CLI
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig,
                          PreTrainedTokenizerFast, get_constant_schedule_with_warmup, get_cosine_schedule_with_warmup,
                          get_cosine_with_min_lr_schedule_with_warmup, get_linear_schedule_with_warmup,
                          get_wsd_schedule)

# Dependencias opcionales: si faltan, el notebook sigue y desactiva lo que dependa de ellas.
try:
    import triton  # torch.compile en GPU
except ImportError:
    triton = None
try:
    import bitsandbytes  # QLoRA
    ERROR_BITSANDBYTES = None
except Exception as e:  # noqa: BLE001  (ImportError, o RuntimeError si no encuentra sus binarios)
    bitsandbytes, ERROR_BITSANDBYTES = None, f"{type(e).__name__}: {e}"
try:
    from google.colab import userdata as colab_userdata  # solo existe en Colab
except ImportError:
    colab_userdata = None
try:
    from mlflow import MlflowClient  # registro en MLflow, solo con MLFLOW_IP (pip install mlflow-skinny)
    from mlflow.entities import Metric, Param, RunTag
except ImportError:
    MlflowClient = Metric = Param = RunTag = None

transformers.utils.logging.disable_progress_bar()

# Versiones por debajo de las cuales el notebook no funciona: torch 2.6 lo exige transformers 5, transformers 4.56
# introdujo el argumento dtype de from_pretrained y peft 0.16 trae LoRA-FA. No se fija ninguna versión exacta.
VERSIONES_MINIMAS = {"torch": "2.6", "transformers": "4.56", "peft": "0.16"}


def titulo(texto: str) -> None:
    """Imprime un encabezado de sección para separar las etapas en la salida."""
    print(f"\n== {texto} " + "=" * max(3, 70 - len(texto)))


def en_colab() -> bool:
    """Devuelve True si el notebook corre en un runtime de Google Colab, desde la interfaz web o desde la CLI."""
    return "COLAB_RELEASE_TAG" in os.environ or colab_userdata is not None


def versiones() -> dict:
    """Devuelve la versión de Python y de cada librería del notebook (None si no está instalada)."""
    v = {"python": platform.python_version(), "torch": torch.__version__, "cuda_de_torch": torch.version.cuda,
         "transformers": transformers.__version__, "peft": peft.__version__}
    for paquete in ("accelerate", "bitsandbytes", "triton", "scikit-learn", "huggingface-hub", "mlflow", "mlflow-skinny"):
        try:
            v[paquete] = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            v[paquete] = None
    return v


def verificar_versiones() -> dict:
    """Imprime las versiones instaladas y avisa de las que están por debajo de VERSIONES_MINIMAS.

    No instala ni fija nada: en Colab se usan las versiones que trae el runtime.
    """
    v = versiones()
    print("  " + " · ".join(f"{k} {val}" for k, val in v.items() if val))
    for paquete, minima in VERSIONES_MINIMAS.items():
        if Version(v[paquete]) < Version(minima):
            print(f"AVISO: {paquete} {v[paquete]} es anterior a {minima}, la mínima que necesita el notebook.")
    return v


def leer_hf_token() -> tuple[str | None, str]:
    """Busca el token de Hugging Face y devuelve (token, origen). El origen es un texto para el registro.

    - En Colab lo lee del secret HF_TOKEN (panel de la llave en la interfaz web, con acceso activado para el
      notebook). Los secrets solo los entrega la interfaz web: desde la CLI la petición se agota a los 10 s.
      Si el secret no está disponible, usa la variable de entorno HF_TOKEN (con la CLI: --env HF_TOKEN=...).
    - En local lo lee de la variable de entorno HF_TOKEN.
    """
    token_entorno = os.environ.get("HF_TOKEN", "").strip()
    if not (en_colab() and colab_userdata is not None):
        return (token_entorno, "variable de entorno HF_TOKEN") if token_entorno else (None, "no hay variable de entorno HF_TOKEN")
    try:
        token = (colab_userdata.get("HF_TOKEN") or "").strip()
        if token:
            return token, "secret HF_TOKEN de Colab"
        motivo = "el secret HF_TOKEN está vacío"
    except Exception as e:  # noqa: BLE001  (las clases de error de google.colab cambian entre versiones)
        motivo = {
            "SecretNotFoundError": "no existe el secret HF_TOKEN",
            "NotebookAccessError": "el notebook no tiene acceso al secret HF_TOKEN (actívalo en el panel de secrets)",
            "TimeoutException": "el secret no respondió: Colab solo entrega secrets a la interfaz web, no a la CLI",
        }.get(type(e).__name__, f"no se pudo leer el secret HF_TOKEN ({type(e).__name__}: {e})")
    if token_entorno:
        return token_entorno, f"variable de entorno HF_TOKEN ({motivo})"
    return None, motivo


def configurar_hf_token() -> str:
    """Deja el token listo para las descargas de Hugging Face y devuelve de dónde salió. Nunca lo imprime.

    Con token, lo pone en la variable de entorno HF_TOKEN, que huggingface_hub consulta antes que cualquier otra
    fuente. Sin token en Colab, desactiva la búsqueda implícita de huggingface_hub para que no vuelva a pedir el
    secret por su cuenta (otros 10 s de espera y un aviso); los modelos públicos se descargan igual.
    """
    token, origen = leer_hf_token()
    if token:
        os.environ["HF_TOKEN"] = token
        print(f"  token de Hugging Face: {origen}")
    else:
        if en_colab():
            hf_constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True
        print(f"  sin token de Hugging Face ({origen}): solo se pueden descargar modelos públicos")
    return origen


titulo("Entorno")
VERSIONES = verificar_versiones()
ORIGEN_HF_TOKEN = configurar_hf_token()

In [ ]:
# @title 2 · Configuración
SCHEDULERS = ("linear", "cosine", "cosine_min", "constant", "wsd", "plateau")  # planificadores de build_scheduler


@dataclass
class Config:
    """Parámetros del experimento. Cada campo acepta una variable de entorno con su nombre en mayúsculas."""

    # Ejecución
    modo: str = "auto"                  # auto | mini | completo
    nombre_corrida: str = ""            # vacío: se arma con modelo, variante, configuración y hora
    semilla: int = 42
    # Datos
    datos_dir: str = ""                 # carpeta con train/dev/test.jsonl o un .zip; vacío: se busca
    configuracion: str = "A"            # A: título + abstract del citado · B: top-3 fragmentos del citado
    incluir_seccion: bool = True        # antepone la sección retórica del citante, como el baseline SciBERT
    max_longitud: int = 1024            # tokens; la configuración B llega a ~800 con el tokenizador de SmolLM2
    evaluar_test: bool = True          # test se usa una sola vez, con el modelo finalista
    # Modelo
    modelo: str = "HuggingFaceTB/SmolLM2-135M"   # id de Hugging Face; "humo": modelo diminuto aleatorio, sin descargas
    # Adaptador
    lora_tipo: str = "lora"             # ver listar_variantes()
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_modulos: str = "auto"          # auto: todas las capas lineales salvo la cabeza; o "q_proj,v_proj"
    # Entrenamiento
    epocas: int = 1
    lr: float = 2e-4
    batch: int = 16
    batch_eval: int = 32
    acumulacion: int = 1
    weight_decay: float = 0.0
    warmup: float = 0.05                # fracción de pasos con calentamiento lineal; después manda `scheduler`
    scheduler: str = "linear"           # linear | cosine | cosine_min | constant | wsd | plateau (ver build_scheduler)
    max_grad_norm: float = 1.0
    max_pasos: int = 0                  # >0 corta el entrenamiento tras N pasos de optimizador
    pesos_clase: str = "ninguno"        # ninguno | inverso | raiz_inversa
    log_cada: int = 10                  # pasos de optimizador entre líneas de registro
    barra_progreso: bool = True         # barra de progreso del entrenamiento
    # Early stopping por macro-F1 de dev
    evals_por_epoca: int = 2            # evaluaciones de dev por época, repartidas en partes iguales
    paciencia: int = 3                  # evaluaciones seguidas sin mejora antes de parar; 0 = nunca para
    delta_minimo: float = 0.0           # mejora mínima de macro-F1 (escala 0-1) que cuenta como mejora
    # Optimizaciones (auto: según el hardware)
    precision: str = "auto"             # auto | bf16 | fp16 | fp32
    compilar: str = "no"                # no | si | auto (si = auto: compila si el hardware lo soporta).
                                        # Compilar cuesta minutos al inicio: solo compensa con modelos grandes y corridas largas.
    modo_compilacion: str = "default"   # default | max-autotune-no-cudagraphs (compila más lento, kernels más rápidos)
    atencion: str = "sdpa"              # sdpa | eager | flash_attention_2 (requiere el paquete flash-attn)
    grad_checkpointing: bool = False    # ahorra memoria a cambio de ~30 % más de tiempo
    num_workers: int = 0                # los datos ya van tokenizados; 0 basta
    # Salidas
    salidas_dir: str = ""               # vacío: /content/salidas en Colab, ./salidas en local
    guardar: bool = True
    # MLflow (vacío: no se usa). La corrida se llama como nombre_corrida; ver RegistroMLflow, celda 11
    mlflow_ip: str = ""                 # ip:puerto del servidor, por ejemplo 54.225.26.128:8080
    mlflow_experimento: str = "baseline_lora"
    mlflow_espera_min: float = 20.0     # si el servidor cae, al terminar se lo espera hasta N min desde la caída
    mlflow_adaptador: bool = False      # sube también los pesos del adaptador (unos 175 MB con 8B y r=16)
    # Ajustes del modo mini
    mini_train: int = 96
    mini_dev: int = 64
    mini_max_longitud: int = 256
    mini_max_pasos: int = 12


def _convertir(valor, tipo):
    """Convierte el texto de una variable de entorno o de CONFIG_JSON al tipo del campo de Config."""
    if tipo is bool:
        return str(valor).strip().lower() in ("1", "true", "si", "sí", "yes", "y")
    if tipo is int:
        return int(float(valor))
    if tipo is float:
        return float(valor)
    return str(valor)


def construir_config(**cambios) -> Config:
    """Arma la Config con esta prioridad: valores por defecto < variables de entorno < CONFIG_JSON < argumentos."""
    cfg = Config()
    tipos = {f.name: f.type for f in fields(Config)}
    for nombre, tipo in tipos.items():
        valor = os.environ.get(nombre.upper())
        if valor not in (None, ""):
            setattr(cfg, nombre, _convertir(valor, tipo))
    if os.environ.get("CONFIG_JSON"):
        for nombre, valor in json.loads(os.environ["CONFIG_JSON"]).items():
            if nombre not in tipos:
                raise KeyError(f"CONFIG_JSON: campo desconocido '{nombre}'. Campos: {', '.join(tipos)}")
            setattr(cfg, nombre, _convertir(valor, tipos[nombre]))
    for nombre, valor in cambios.items():
        if nombre not in tipos:
            raise KeyError(f"Campo desconocido '{nombre}'. Campos: {', '.join(tipos)}")
        setattr(cfg, nombre, _convertir(valor, tipos[nombre]))
    return resolver_config(cfg)


def resolver_config(cfg: Config) -> Config:
    """Valida los valores, resuelve los automáticos (modo, carpeta de salida, nombre) y aplica los límites del modo mini."""
    cfg.configuracion = cfg.configuracion.upper()
    if cfg.configuracion not in ("A", "B"):
        raise ValueError(f"CONFIGURACION debe ser A o B, no '{cfg.configuracion}'")
    if cfg.modo == "auto":
        cfg.modo = "completo" if (en_colab() and torch.cuda.is_available()) else "mini"
    if cfg.modo not in ("mini", "completo"):
        raise ValueError(f"MODO debe ser auto, mini o completo, no '{cfg.modo}'")
    if cfg.modo_compilacion not in ("default", "max-autotune-no-cudagraphs", "max-autotune", "reduce-overhead"):
        raise ValueError(f"MODO_COMPILACION no válido: '{cfg.modo_compilacion}'")
    if cfg.pesos_clase not in ("ninguno", "inverso", "raiz_inversa"):
        raise ValueError(f"PESOS_CLASE debe ser ninguno, inverso o raiz_inversa, no '{cfg.pesos_clase}'")
    if cfg.scheduler not in SCHEDULERS:
        raise ValueError(f"SCHEDULER debe ser uno de {', '.join(SCHEDULERS)}, no '{cfg.scheduler}'")
    if cfg.evals_por_epoca < 1:
        raise ValueError(f"EVALS_POR_EPOCA debe ser 1 o más, no {cfg.evals_por_epoca}")
    if cfg.paciencia < 0 or cfg.delta_minimo < 0:
        raise ValueError("PACIENCIA y DELTA_MINIMO no pueden ser negativos")
    if cfg.mlflow_espera_min < 0:
        raise ValueError("MLFLOW_ESPERA_MIN no puede ser negativo")
    if cfg.modo == "mini":
        cfg.max_longitud = min(cfg.max_longitud, cfg.mini_max_longitud)
        cfg.max_pasos = cfg.max_pasos or cfg.mini_max_pasos
        cfg.batch = min(cfg.batch, 8)
        cfg.batch_eval = min(cfg.batch_eval, 16)
        cfg.log_cada = min(cfg.log_cada, 2)
    if not cfg.salidas_dir:
        cfg.salidas_dir = "/content/salidas" if en_colab() else "salidas"
    if not cfg.nombre_corrida:
        corto = cfg.modelo.rstrip("/").split("/")[-1]
        cfg.nombre_corrida = f"{corto}-{cfg.lora_tipo}-cfg{cfg.configuracion}-{cfg.modo}-{datetime.now():%Y%m%d-%H%M%S}"
    return cfg


def imprimir_config(cfg: Config) -> None:
    """Imprime los campos de la Config que más cambian entre corridas."""
    claves = ("modo", "modelo", "lora_tipo", "configuracion", "max_longitud", "epocas", "lr", "batch",
              "acumulacion", "scheduler", "lora_r", "lora_alpha", "evals_por_epoca", "paciencia", "precision", "compilar",
              "max_pasos", "salidas_dir", "nombre_corrida")
    for k in claves:
        print(f"  {k:<16}{getattr(cfg, k)}")

In [ ]:
# @title 3 · Hardware y optimizaciones
def detectar_hardware() -> dict:
    """Describe la máquina: sistema, GPU y su capacidad de cómputo (cc), memoria y librerías opcionales disponibles."""
    hw = {
        "sistema": platform.system(),
        "colab": en_colab(),
        "nucleos_cpu": os.cpu_count(),
        "dispositivo": "cpu",
        "gpu": None,
        "cc": None,
        "vram_gb": None,
        "triton": triton is not None,
        "compilador_c": any(shutil.which(c) for c in ("gcc", "cc", "clang")),
        "bitsandbytes": bitsandbytes is not None,
    }
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        hw.update(dispositivo="cuda", gpu=p.name, cc=list(torch.cuda.get_device_capability(0)),
                  vram_gb=round(p.total_memory / 1024**3, 1), n_gpus=torch.cuda.device_count())
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        hw["dispositivo"] = "mps"
    return hw


def decidir_optimizaciones(hw: dict, cfg: Config) -> dict:
    """Decide cada optimización según el hardware y la Config. Devuelve la decisión y el motivo de cada una."""
    cuda = hw["dispositivo"] == "cuda"
    cc = tuple(hw["cc"] or (0, 0))
    ampere = cuda and cc >= (8, 0)  # A100 (8.0), L4 (8.9), H100 (9.0), G4 / RTX PRO 6000 (12.0)
    motivos = {}

    # Precisión mixta
    if cfg.precision == "auto":
        precision = "bf16" if ampere else ("fp16" if cuda else "fp32")
        motivos["precision"] = ("bf16 nativo (cc ≥ 8.0)" if ampere
                                else "GPU sin bf16 nativo: fp16 con GradScaler" if cuda
                                else f"{hw['dispositivo'].upper()}: fp32")
    else:
        precision = cfg.precision
        motivos["precision"] = "fijada por configuración"
        if precision == "bf16" and not ampere:
            precision = "fp16" if cuda else "fp32"
            motivos["precision"] = f"bf16 pedido pero la GPU no lo soporta: {precision}"
        elif precision == "fp16" and not cuda:
            precision = "fp32"
            motivos["precision"] = "fp16 pedido sin CUDA: fp32"
    dtype = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}[precision]

    # TF32 para las multiplicaciones que quedan en fp32
    tf32 = ampere
    motivos["tf32"] = "multiplicaciones fp32 en tensor cores" if tf32 else "requiere GPU con cc ≥ 8.0"

    # torch.compile (inductor + triton)
    requisitos = (("CUDA", cuda), ("triton", hw["triton"]), ("compilador C", hw["compilador_c"]),
                  ("Linux", hw["sistema"] == "Linux"), ("cc ≥ 7.0", not cuda or cc >= (7, 0)))
    faltan = [nombre for nombre, ok in requisitos if not ok]
    if cfg.compilar == "no":
        compilar, motivos["compilar"] = False, "desactivado por configuración"
    elif faltan:
        compilar, motivos["compilar"] = False, "no soportado aquí: falta " + ", ".join(faltan)
    else:
        compilar, motivos["compilar"] = True, "inductor + triton; si falla, sigue sin compilar"

    motivos["adamw_fusionado"] = "kernel único de CUDA para todos los parámetros" if cuda else "solo en CUDA"
    motivos["pin_memory"] = "copias asíncronas CPU → GPU" if cuda else "solo en CUDA"
    motivos["multiplo_padding"] = ("64: menos formas distintas que recompilar" if compilar
                                   else "8: alinea con tensor cores")
    motivos["atencion"] = ("SDPA: PyTorch elige flash, memory-efficient o cuDNN según la GPU" if cfg.atencion == "sdpa"
                           else "fijada por configuración")
    cuant = cuda and hw["bitsandbytes"]
    motivos["cuant_4bit"] = ("disponible para LORA_TIPO=qlora" if cuant
                             else "requiere GPU NVIDIA" if hw["bitsandbytes"]
                             else f"requiere bitsandbytes ({(ERROR_BITSANDBYTES or 'no está instalado')[:120]})")
    return {
        "dispositivo": hw["dispositivo"], "precision": precision, "dtype": dtype,
        "autocast": precision != "fp32", "escalador": precision == "fp16", "tf32": tf32,
        "compilar": compilar, "adamw_fusionado": cuda, "pin_memory": cuda, "non_blocking": cuda,
        "multiplo_padding": 64 if compilar else 8, "atencion": cfg.atencion, "cuant_4bit": cuant,
        "motivos": motivos,
    }


def aplicar_optimizaciones_globales(opt: dict) -> None:
    """Aplica los ajustes globales de torch: TF32 en GPUs con cc ≥ 8.0 y más recompilaciones permitidas al compilar.

    TF32 se activa con set_float32_matmul_precision, que mantiene coherentes la API clásica y la nueva de torch ≥ 2.9.
    Fijar solo la nueva (fp32_precision) deja a torch.get_float32_matmul_precision() lanzando un error, y
    torch.compile lo consulta al autoajustar multiplicaciones.
    """
    if opt["tf32"]:
        torch.set_float32_matmul_precision("high")
    if opt["compilar"]:
        # Las longitudes de lote varían: se permiten más recompilaciones antes de volver a eager.
        dynamo = torch._dynamo.config
        limite = "recompile_limit" if hasattr(dynamo, "recompile_limit") else "cache_size_limit"  # nombre previo a torch 2.7
        setattr(dynamo, limite, max(getattr(dynamo, limite), 32))


def contexto_autocast(opt: dict):
    """Devuelve el contexto de precisión mixta (bf16 o fp16) del dispositivo, o uno vacío en fp32."""
    if opt["autocast"]:
        return torch.autocast(device_type=opt["dispositivo"], dtype=opt["dtype"])
    return contextlib.nullcontext()


def imprimir_decisiones(hw: dict, opt: dict) -> None:
    """Imprime el hardware detectado y, para cada optimización, si quedó activa y por qué."""
    if hw["dispositivo"] == "cuda":
        print(f"  {hw['gpu']} · cc {hw['cc'][0]}.{hw['cc'][1]} · {hw['vram_gb']} GB · {hw['sistema']}"
              + (" · Colab" if hw["colab"] else ""))
    else:
        print(f"  {hw['dispositivo'].upper()} · {hw['nucleos_cpu']} núcleos · {hw['sistema']}"
              + (" · Colab" if hw["colab"] else ""))
    filas = (("precisión", opt["precision"], "precision"), ("TF32", opt["tf32"], "tf32"),
             ("torch.compile", opt["compilar"], "compilar"), ("AdamW fusionado", opt["adamw_fusionado"], "adamw_fusionado"),
             ("pin_memory", opt["pin_memory"], "pin_memory"), ("padding múltiplo", opt["multiplo_padding"], "multiplo_padding"),
             ("atención", opt["atencion"], "atencion"), ("QLoRA 4-bit", opt["cuant_4bit"], "cuant_4bit"))
    for nombre, valor, clave in filas:
        valor = {True: "sí", False: "no"}.get(valor, valor)
        print(f"  {nombre:<18}{str(valor):<7}{opt['motivos'][clave]}")

In [ ]:
# @title 4 · Datos
# Mismo orden que CANONICAL_9_CLASSES en src/data/builder.py del repo EvidenCite.
ETIQUETAS = ["Background", "Uses", "Comparison", "Extends", "Motivation",
             "Future Work", "Evidence", "Originator", "Further Reading"]
ETIQ2ID = {e: i for i, e in enumerate(ETIQUETAS)}
SPLITS = ("train", "dev", "test")
CAMPOS_OBLIGATORIOS = ("instance_id", "label", "annotation_metadata", "citing_paper", "citation_context", "cited_paper")


def _tiene_splits(carpeta: Path) -> bool:
    """True si la carpeta contiene train.jsonl, dev.jsonl y test.jsonl."""
    return all((carpeta / f"{s}.jsonl").is_file() for s in SPLITS)


def _extraer_zip(ruta_zip: Path, destino: Path) -> None:
    """Extrae el zip en `destino`, omitiendo carpetas ocultas y de macOS y rechazando rutas que salgan del destino."""
    destino = destino.resolve()
    with zipfile.ZipFile(ruta_zip) as z:
        for miembro in z.infolist():
            partes = Path(miembro.filename).parts
            if not partes or partes[0] == "__MACOSX" or any(p.startswith(".") for p in partes):
                continue
            objetivo = (destino / miembro.filename).resolve()
            if not objetivo.is_relative_to(destino):
                raise ValueError(f"Ruta insegura dentro del zip: {miembro.filename}")
            z.extract(miembro, destino)


def _buscar_en_zip(ruta_zip: Path) -> Path | None:
    """Extrae el zip junto a sí mismo y devuelve la primera carpeta que tenga los tres splits, o None."""
    destino = ruta_zip.parent / f"{ruta_zip.stem}_extraido"
    _extraer_zip(ruta_zip, destino)
    for carpeta in [destino, *sorted(p for p in destino.rglob("*") if p.is_dir())]:
        if _tiene_splits(carpeta):
            return carpeta
    return None


def localizar_datos(cfg: Config) -> Path:
    """Encuentra la carpeta del dataset: DATOS_DIR si está definido; si no, rutas habituales y dataset.zip."""
    if cfg.datos_dir:
        ruta = Path(cfg.datos_dir)
        if ruta.is_dir() and _tiene_splits(ruta):
            return ruta
        if ruta.is_file() and ruta.suffix == ".zip" and (encontrada := _buscar_en_zip(ruta)):
            return encontrada
        raise FileNotFoundError(f"DATOS_DIR={cfg.datos_dir} no contiene train/dev/test.jsonl")
    for c in ("data/dataset", "dataset", "../data/dataset", "/content/data/dataset", "/content/dataset"):
        if _tiene_splits(Path(c)):
            return Path(c)
    for z in ("dataset.zip", "data/dataset.zip", "/content/dataset.zip"):
        if Path(z).is_file() and (encontrada := _buscar_en_zip(Path(z))):
            return encontrada
    raise FileNotFoundError(
        "No encontré train/dev/test.jsonl. Pon la carpeta en data/dataset/, deja dataset.zip junto al notebook "
        "(en Colab: colab upload -s <sesión> dataset.zip /content/dataset.zip) o define DATOS_DIR.")


def leer_jsonl(ruta: Path) -> list[dict]:
    """Lee un archivo JSON Lines y devuelve una lista de diccionarios (ignora líneas vacías)."""
    with open(ruta, encoding="utf-8") as f:
        return [json.loads(linea) for linea in f if linea.strip()]


def validar_instancias(filas: list[dict], split: str) -> None:
    """Comprueba que cada instancia tenga los campos obligatorios y una etiqueta conocida; si no, lanza ValueError."""
    errores = []
    for f in filas:
        faltan = [c for c in CAMPOS_OBLIGATORIOS if c not in f]
        if faltan:
            errores.append(f"{f.get('instance_id', '?')}: faltan {faltan}")
        elif f["label"] not in ETIQ2ID:
            errores.append(f"{f['instance_id']}: etiqueta desconocida '{f['label']}'")
    if errores:
        raise ValueError(f"{split}: {len(errores)} instancias inválidas. Primeras: {errores[:5]}")


def submuestra_estratificada(filas: list[dict], n: int, semilla: int) -> list[dict]:
    """Toma n instancias repartidas por clase (round-robin), para que el modo mini vea todas las clases."""
    if n <= 0 or n >= len(filas):
        return list(filas)
    rng = random.Random(semilla)
    por_clase: dict[str, list] = {}
    for f in filas:
        por_clase.setdefault(f["label"], []).append(f)
    for lista in por_clase.values():
        rng.shuffle(lista)
    salida = []
    while len(salida) < n:
        for etiqueta in sorted(por_clase):
            if por_clase[etiqueta] and len(salida) < n:
                salida.append(por_clase[etiqueta].pop())
    rng.shuffle(salida)
    return salida


def resumen_split(filas: list[dict]) -> dict:
    """Cuenta instancias por clase y por fuente de un split."""
    return {"n": len(filas),
            "clases": dict(Counter(f["label"] for f in filas).most_common()),
            "fuentes": dict(Counter(f["annotation_metadata"].get("source_dataset", "?") for f in filas).most_common())}


def cargar_datos(cfg: Config) -> dict:
    """Carga y valida train, dev y test; en modo mini toma submuestras estratificadas. Imprime un resumen."""
    carpeta = localizar_datos(cfg)
    datos = {}
    for s in SPLITS:
        filas = leer_jsonl(carpeta / f"{s}.jsonl")
        validar_instancias(filas, s)
        datos[s] = filas
    if cfg.modo == "mini":
        datos["train"] = submuestra_estratificada(datos["train"], cfg.mini_train, cfg.semilla)
        datos["dev"] = submuestra_estratificada(datos["dev"], cfg.mini_dev, cfg.semilla)
        datos["test"] = submuestra_estratificada(datos["test"], cfg.mini_dev, cfg.semilla)
    print(f"  carpeta: {carpeta}")
    for s in SPLITS:
        r = resumen_split(datos[s])
        clases = ", ".join(f"{k} {v}" for k, v in r["clases"].items())
        fuentes = ", ".join(f"{k} {v}" for k, v in r["fuentes"].items())
        print(f"  {s:<6}{r['n']:>5} · {clases} · fuentes: {fuentes}")
    return datos

In [ ]:
# @title 5 · Texto de entrada y tokenización
# La cabeza lee el último token, así que la secuencia termina en una pregunta fija y lo que se recorta es la
# evidencia del citado, nunca el contexto ni la pregunta. El texto va en inglés, como el preentrenamiento.
PLANTILLA = {
    "citante": "Citing section: {seccion}\nCitation context: {contexto}\n",
    "citante_sin_seccion": "Citation context: {contexto}\n",
    "citado_A": "Cited paper: {titulo}. {abstract}\n",
    "citado_B": "Cited paper excerpts:\n{fragmentos}\n",
    "fragmento": "[{seccion}] {texto}",
    "pregunta": "Citation function:",
}


def partes_texto(inst: dict, cfg: Config) -> tuple[str, str]:
    """Devuelve (texto del citante, texto del citado) según la configuración A o B."""
    contexto = inst["citation_context"].strip()
    seccion = (inst["citing_paper"].get("rhetorical_section") or "").strip()
    if cfg.incluir_seccion and seccion:
        citante = PLANTILLA["citante"].format(seccion=seccion, contexto=contexto)
    else:
        citante = PLANTILLA["citante_sin_seccion"].format(contexto=contexto)
    citado = inst["cited_paper"]
    if cfg.configuracion == "A":
        texto_citado = PLANTILLA["citado_A"].format(titulo=(citado.get("title") or "").strip(),
                                                   abstract=(citado.get("abstract") or "").strip())
    else:
        fragmentos = sorted(citado.get("retrieved_chunks") or [], key=lambda c: c.get("rank", 0))
        texto_citado = PLANTILLA["citado_B"].format(fragmentos="\n".join(
            PLANTILLA["fragmento"].format(seccion=(c.get("section_heading") or "").strip(), texto=c["text"].strip())
            for c in fragmentos))
    return citante, texto_citado


def _especiales_iniciales(tok) -> list[int]:
    """Tokens que el tokenizador antepone por su cuenta (por ejemplo BOS en Llama 3)."""
    con = tok("a")["input_ids"]
    sin = tok("a", add_special_tokens=False)["input_ids"]
    for i in range(len(con) - len(sin) + 1):
        if con[i:i + len(sin)] == sin:
            return con[:i]
    return []


def tokenizar_split(filas: list[dict], tok, cfg: Config) -> list[dict]:
    """Tokeniza un split: [especiales iniciales] + citante + citado + pregunta, sin pasar de max_longitud.

    Si no cabe, recorta el texto del citado; si el contexto solo ya excede el presupuesto, conserva su final.
    Cada ítem guarda su índice, ids, etiqueta, id de instancia, fuente y si se recortó.
    """
    partes = [partes_texto(f, cfg) for f in filas]
    ids_citante = tok([p[0] for p in partes], add_special_tokens=False)["input_ids"]
    ids_citado = tok([p[1] for p in partes], add_special_tokens=False)["input_ids"]
    ids_pregunta = tok(PLANTILLA["pregunta"], add_special_tokens=False)["input_ids"]
    inicio = _especiales_iniciales(tok)
    presupuesto = cfg.max_longitud - len(inicio) - len(ids_pregunta)
    items = []
    for k, (f, a, b) in enumerate(zip(filas, ids_citante, ids_citado)):
        truncado = len(a) + len(b) > presupuesto
        if len(a) > presupuesto:          # contexto excepcionalmente largo: se conserva su final
            a, b = a[-presupuesto:], []
        elif truncado:
            b = b[:presupuesto - len(a)]
        items.append({"indice": k, "input_ids": inicio + a + b + ids_pregunta, "etiqueta": ETIQ2ID[f["label"]],
                      "instance_id": f["instance_id"], "fuente": f["annotation_metadata"].get("source_dataset", "?"),
                      "truncado": truncado})
    return items


def resumen_tokens(items: list[dict]) -> dict:
    """Estadísticas de longitud en tokens de un split: media, p95, máximo y % de ejemplos recortados."""
    largos = np.array([len(x["input_ids"]) for x in items])
    return {"n": len(items), "media": float(largos.mean()), "p95": float(np.percentile(largos, 95)),
            "max": int(largos.max()), "truncados_pct": 100.0 * float(np.mean([x["truncado"] for x in items]))}

In [ ]:
# @title 6 · Lotes
class LotesPorLongitud(torch.utils.data.Sampler):
    """Agrupa ejemplos de longitud parecida para rellenar menos.

    Entrenamiento: baraja, ordena por longitud dentro de bloques de `factor` lotes y baraja los lotes; el lote más
    largo va primero para que un problema de memoria aparezca en el paso 1. Evaluación: ordena por longitud.
    """

    def __init__(self, longitudes, tam_lote: int, mezclar: bool, semilla: int = 0, factor: int = 50):
        """Guarda las longitudes de los ejemplos, el tamaño de lote y si hay que barajar (con qué semilla)."""
        super().__init__()
        self.longitudes, self.tam, self.mezclar, self.semilla, self.factor = list(longitudes), tam_lote, mezclar, semilla, factor
        self.epoca = 0

    def __iter__(self):
        """Genera las listas de índices de cada lote; en entrenamiento, con un orden distinto en cada época."""
        idx = list(range(len(self.longitudes)))
        if not self.mezclar:
            idx.sort(key=lambda i: self.longitudes[i], reverse=True)
            yield from (idx[i:i + self.tam] for i in range(0, len(idx), self.tam))
            return
        rng = random.Random(self.semilla + self.epoca)
        self.epoca += 1
        rng.shuffle(idx)
        lotes = []
        bloque = self.tam * self.factor
        for i in range(0, len(idx), bloque):
            orden = sorted(idx[i:i + bloque], key=lambda j: self.longitudes[j], reverse=True)
            lotes += [orden[k:k + self.tam] for k in range(0, len(orden), self.tam)]
        rng.shuffle(lotes)
        mayor = max(range(len(lotes)), key=lambda k: max(self.longitudes[j] for j in lotes[k]))
        lotes[0], lotes[mayor] = lotes[mayor], lotes[0]
        yield from lotes

    def __len__(self):
        """Número de lotes por época."""
        return math.ceil(len(self.longitudes) / self.tam)


class DatosTokenizados(torch.utils.data.Dataset):
    """Dataset de PyTorch sobre la lista de ítems ya tokenizados."""

    def __init__(self, items: list[dict]):
        """Guarda la lista de ítems."""
        self.items = items

    def __len__(self):
        """Número de ítems."""
        return len(self.items)

    def __getitem__(self, i):
        """Devuelve el ítem i tal cual; el relleno lo hace colar()."""
        return self.items[i]


def colar(lote: list[dict], pad_id: int, multiplo: int) -> dict:
    """Arma los tensores del lote con relleno dinámico a la derecha hasta el múltiplo de `multiplo` más cercano."""
    largo = max(len(x["input_ids"]) for x in lote)
    largo = ((largo + multiplo - 1) // multiplo) * multiplo
    ids = torch.full((len(lote), largo), pad_id, dtype=torch.long)
    mascara = torch.zeros((len(lote), largo), dtype=torch.long)
    for i, x in enumerate(lote):
        n = len(x["input_ids"])
        ids[i, :n] = torch.tensor(x["input_ids"], dtype=torch.long)
        mascara[i, :n] = 1
    return {"input_ids": ids, "attention_mask": mascara,
            "labels": torch.tensor([x["etiqueta"] for x in lote], dtype=torch.long),
            "indices": torch.tensor([x["indice"] for x in lote], dtype=torch.long)}


def crear_dataloader(items: list[dict], tam_lote: int, mezclar: bool, cfg: Config, opt: dict, pad_id: int):
    """Crea el DataLoader de un split con lotes agrupados por longitud y relleno dinámico."""
    muestreador = LotesPorLongitud([len(x["input_ids"]) for x in items], tam_lote, mezclar, cfg.semilla)
    return torch.utils.data.DataLoader(
        DatosTokenizados(items), batch_sampler=muestreador,
        collate_fn=functools.partial(colar, pad_id=pad_id, multiplo=opt["multiplo_padding"]),
        num_workers=cfg.num_workers, pin_memory=opt["pin_memory"], persistent_workers=cfg.num_workers > 0)


def mover(lote: dict, opt: dict) -> dict:
    """Mueve los tensores del lote al dispositivo de entrenamiento (de forma asíncrona en CUDA)."""
    return {k: v.to(opt["dispositivo"], non_blocking=opt["non_blocking"]) if torch.is_tensor(v) else v
            for k, v in lote.items()}

In [ ]:
# @title 7 · Modelo
CABEZAS = ("score", "classifier")  # nombres de la cabeza de clasificación en transformers


def tokenizador_humo(textos: list[str]):
    """Entrena un tokenizador BPE de 4000 piezas sobre `textos` para el modelo de humo, sin descargar nada."""
    bpe = Tokenizer(BPE(unk_token="<unk>"))
    bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    bpe.decoder = decoders.ByteLevel()
    bpe.train_from_iterator(textos, trainers.BpeTrainer(
        vocab_size=4000, special_tokens=["<pad>", "<unk>", "<s>", "</s>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
    return PreTrainedTokenizerFast(tokenizer_object=bpe, pad_token="<pad>", unk_token="<unk>",
                                   bos_token="<s>", eos_token="</s>")


def cargar_tokenizer(ruta: str):
    """Carga el tokenizador del modelo, asegura un token de relleno y pone el relleno a la derecha.

    Si el modelo no trae pad, usa EOS. Es seguro porque el texto nunca contiene EOS: verificar_pooling comprueba
    que la cabeza siga leyendo el último token real.
    """
    tok = AutoTokenizer.from_pretrained(ruta, token=os.environ.get("HF_TOKEN") or None)
    if tok.pad_token_id is None:
        if tok.eos_token is None:
            raise ValueError("El tokenizador no tiene pad ni eos: define tokenizer.pad_token a mano.")
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok


def argumentos_modelo(opt: dict, var) -> dict:
    """Argumentos de from_pretrained: 9 etiquetas, dtype de la precisión elegida, atención y token.

    En QLoRA añade la cuantización NF4 de la base, con la cabeza sin cuantizar y todo el modelo en la GPU 0.
    """
    kwargs = dict(num_labels=len(ETIQUETAS), id2label=dict(enumerate(ETIQUETAS)), label2id=ETIQ2ID,
                  dtype=opt["dtype"], attn_implementation=opt["atencion"], token=os.environ.get("HF_TOKEN") or None)
    if var.cuantizar_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=opt["dtype"], llm_int8_skip_modules=list(CABEZAS))
        kwargs["device_map"] = {"": 0}
    return kwargs


def crear_modelo(cfg: Config, opt: dict, var, textos_humo: list[str] | None = None):
    """Crea el tokenizador y el modelo de clasificación a partir de `cfg.modelo`. Devuelve (modelo, tokenizador, ruta).

    Para cambiar de modelo basta con cambiar `cfg.modelo` (variable MODELO) por otro id de Hugging Face.
    Esta es la única función con imports dentro: las clases propias de una familia de modelos se importan aquí,
    solo cuando se usan (por ejemplo, las de Llama para el modelo de humo).

    - "humo": construye un Llama diminuto con pesos aleatorios y un tokenizador BPE entrenado con `textos_humo`,
      lo guarda en disco y lo carga por el mismo camino que un modelo real, sin descargas.
    - Carga AutoModelForSequenceClassification con 9 etiquetas. Si la atención pedida no está disponible, prueba
      sdpa y después eager.
    - Desactiva la caché de KV (solo sirve para generar), mueve el modelo al dispositivo (en QLoRA ya está en la
      GPU) y activa gradient checkpointing si se pidió.
    """
    ruta = cfg.modelo
    if cfg.modelo == "humo":
        from transformers import LlamaConfig, LlamaForCausalLM

        carpeta = Path(cfg.salidas_dir) / "_modelo_humo"
        carpeta.mkdir(parents=True, exist_ok=True)
        tok_humo = tokenizador_humo(textos_humo or ["texto de prueba"])
        tok_humo.save_pretrained(carpeta)
        config = LlamaConfig(vocab_size=len(tok_humo), hidden_size=64, intermediate_size=172, num_hidden_layers=2,
                             num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=4096,
                             pad_token_id=tok_humo.pad_token_id, bos_token_id=tok_humo.bos_token_id,
                             eos_token_id=tok_humo.eos_token_id)
        with torch.random.fork_rng(devices=[]):  # pesos fijos sin alterar la semilla del experimento
            torch.manual_seed(0)
            LlamaForCausalLM(config).save_pretrained(carpeta)
        ruta = str(carpeta)

    tok = cargar_tokenizer(ruta)
    kwargs = argumentos_modelo(opt, var)
    try:
        modelo = AutoModelForSequenceClassification.from_pretrained(ruta, **kwargs)
    except (ValueError, ImportError) as e:
        texto = str(e).lower()
        if not any(k in texto for k in ("attn", "attention", "sdpa", "flash")):
            raise
        respaldo = "sdpa" if opt["atencion"] != "sdpa" else "eager"
        print(f"AVISO: no se pudo usar atención '{opt['atencion']}' ({str(e)[:200]}); uso '{respaldo}'.")
        opt["atencion"] = kwargs["attn_implementation"] = respaldo
        modelo = AutoModelForSequenceClassification.from_pretrained(ruta, **kwargs)
    # En los modelos multimodales (Qwen3.5, Gemma 3 desde 4B) la cabeza y el modelo de texto leen pad_token_id y
    # use_cache de la config de texto, no de la principal; en los de solo texto ambas son la misma.
    for config in (modelo.config, modelo.config.get_text_config()):
        config.pad_token_id = tok.pad_token_id
        if hasattr(config, "use_cache"):
            config.use_cache = False
    if not var.cuantizar_4bit:
        modelo.to(opt["dispositivo"])
    if cfg.grad_checkpointing and not var.cuantizar_4bit:  # en QLoRA lo activa prepare_model_for_kbit_training
        modelo.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return modelo, tok, ruta


def verificar_pooling(lote: dict, pad_id: int) -> None:
    """La cabeza lee el último token distinto de pad. Comprueba que ese sea el último token real de cada fila."""
    ids, mascara = lote["input_ids"], lote["attention_mask"]
    ultimo_real = mascara.sum(1) - 1
    posiciones = torch.arange(ids.shape[1], device=ids.device)
    ultimo_no_pad = (posiciones * (ids != pad_id)).argmax(1)
    if not torch.equal(ultimo_real, ultimo_no_pad):
        raise RuntimeError("El token de relleno aparece dentro del texto y la cabeza leería una posición equivocada. "
                           "Asigna un pad_token que no aparezca en el texto.")

In [ ]:
# @title 8 · Variantes de LoRA
@dataclass
class VarianteLoRA:
    """Describe una variante del adaptador: cómo construir su configuración de PEFT y qué trato especial necesita."""

    descripcion: str
    construir: object                 # función (modelo, cfg) -> configuración de PEFT
    optimizador: str = "adamw"        # adamw | lora_plus | lora_fa
    necesita_datos: bool = False      # EVA: se inicializa con activaciones de lotes de train
    muta_base: bool = False           # PiSSA / OLoRA modifican W; al guardar se convierte a LoRA estándar
    cuantizar_4bit: bool = False      # QLoRA: base en NF4 (solo CUDA + bitsandbytes)


def modulos_lineales(modelo, excluir=CABEZAS) -> list[str]:
    """Nombres (último tramo) de todas las capas lineales salvo la cabeza: el equivalente de 'all-linear'."""
    nombres = set()
    for nombre, modulo in modelo.named_modules():
        hoja = nombre.rsplit(".", 1)[-1]
        if hoja in excluir:
            continue
        if isinstance(modulo, torch.nn.Linear) or type(modulo).__name__ in ("Linear4bit", "Linear8bitLt"):
            nombres.add(hoja)
    return sorted(nombres)


def _modulos(modelo, cfg: Config) -> list[str]:
    """Capas donde va el adaptador: todas las lineales (LORA_MODULOS=auto) o la lista separada por comas."""
    if cfg.lora_modulos == "auto":
        return modulos_lineales(modelo)
    return [m.strip() for m in cfg.lora_modulos.split(",") if m.strip()]


def _lora(modelo, cfg: Config, **extra) -> LoraConfig:
    """LoraConfig de clasificación con r, alpha y dropout de la Config; `extra` activa la variante (DoRA, PiSSA...)."""
    return LoraConfig(task_type=TaskType.SEQ_CLS, r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
                      lora_dropout=cfg.lora_dropout, target_modules=_modulos(modelo, cfg), bias="none", **extra)


def _eva(modelo, cfg: Config) -> LoraConfig:
    """LoraConfig de EVA: inicialización a partir de las activaciones (ver aplicar_lora)."""
    return _lora(modelo, cfg, init_lora_weights="eva", eva_config=EvaConfig(rho=2.0))


def _vera(modelo, cfg: Config):
    """VeraConfig: proyecciones aleatorias compartidas entre capas; solo se entrenan vectores de escala."""
    # El rango suele ser alto porque solo se entrenan vectores.
    return VeraConfig(task_type=TaskType.SEQ_CLS, r=256, target_modules=_modulos(modelo, cfg),
                      vera_dropout=cfg.lora_dropout, d_initial=0.1, save_projection=True)


VARIANTES_LORA = {
    "lora": VarianteLoRA("LoRA estándar: A aleatoria, B en cero", lambda m, c: _lora(m, c)),
    "rslora": VarianteLoRA("rsLoRA: escala α/√r en vez de α/r", lambda m, c: _lora(m, c, use_rslora=True)),
    "dora": VarianteLoRA("DoRA: separa magnitud y dirección de W", lambda m, c: _lora(m, c, use_dora=True)),
    "pissa": VarianteLoRA("PiSSA: inicia con las componentes principales de W (SVD rápida)",
                          lambda m, c: _lora(m, c, init_lora_weights="pissa_niter_4"), muta_base=True),
    "olora": VarianteLoRA("OLoRA: inicia con una base ortonormal (QR de W)",
                          lambda m, c: _lora(m, c, init_lora_weights="olora"), muta_base=True),
    "eva": VarianteLoRA("EVA: inicia con las direcciones de mayor varianza de las activaciones y reparte el rango",
                        _eva, necesita_datos=True),
    "lora_plus": VarianteLoRA("LoRA+: la tasa de B es 16 veces la de A", lambda m, c: _lora(m, c), optimizador="lora_plus"),
    "lora_fa": VarianteLoRA("LoRA-FA: congela A y entrena solo B", lambda m, c: _lora(m, c), optimizador="lora_fa"),
    "vera": VarianteLoRA("VeRA: proyecciones aleatorias compartidas; solo entrena vectores de escala", _vera),
    "qlora": VarianteLoRA("QLoRA: base cuantizada a 4 bits (NF4) + LoRA", lambda m, c: _lora(m, c), cuantizar_4bit=True),
}


def listar_variantes() -> None:
    """Imprime las variantes disponibles con sus particularidades."""
    for nombre, v in VARIANTES_LORA.items():
        notas = [n for n, ok in (("usa lotes de train al iniciar", v.necesita_datos), ("modifica W", v.muta_base),
                                 ("optimizador propio", v.optimizador != "adamw"), ("solo CUDA", v.cuantizar_4bit)) if ok]
        print(f"  {nombre:<10}{v.descripcion}" + (f"  [{', '.join(notas)}]" if notas else ""))


def resolver_variante(cfg: Config, opt: dict):
    """Devuelve la variante pedida en LORA_TIPO y una lista de avisos.

    QLoRA necesita GPU NVIDIA y bitsandbytes; si falta algo, se degrada a LoRA estándar sin cuantizar y lo avisa.
    """
    if cfg.lora_tipo not in VARIANTES_LORA:
        raise KeyError(f"LORA_TIPO='{cfg.lora_tipo}' no existe. Opciones: {', '.join(VARIANTES_LORA)}")
    var, avisos = VARIANTES_LORA[cfg.lora_tipo], []
    if var.cuantizar_4bit and not opt["cuant_4bit"]:
        avisos.append(f"qlora desactivada ({opt['motivos']['cuant_4bit']}); se entrena LoRA estándar sin cuantizar")
        var = dataclasses.replace(var, cuantizar_4bit=False, descripcion=var.descripcion + " · degradada a LoRA")
    return var, avisos


def asegurar_entrenables_fp32(modelo) -> None:
    """Pasa a fp32 los pesos que se optimizan (la cabeza nueva hereda el dtype de la base)."""
    for p in modelo.parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()


def contar_parametros(modelo) -> tuple[int, int]:
    """Devuelve (parámetros entrenables, parámetros totales)."""
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return entrenables, sum(p.numel() for p in modelo.parameters())


def aplicar_lora(modelo, cfg: Config, var: VarianteLoRA, opt: dict, lotes_init=None, dir_init: Path | None = None):
    """Envuelve el modelo con el adaptador de la variante y devuelve (modelo PEFT, información del adaptador).

    - QLoRA: prepara la base cuantizada para entrenar.
    - EVA: inicializa el adaptador con las activaciones de `lotes_init`.
    - PiSSA / OLoRA: guarda el adaptador inicial en `dir_init` para convertirlo a LoRA estándar al final.
    """
    if var.cuantizar_4bit:
        modelo = prepare_model_for_kbit_training(modelo, use_gradient_checkpointing=cfg.grad_checkpointing)
    config_peft = var.construir(modelo, cfg)
    modelo = get_peft_model(modelo, config_peft, low_cpu_mem_usage=var.necesita_datos)  # EVA lo recomienda
    if var.necesita_datos:
        if not lotes_init:
            raise ValueError(f"{cfg.lora_tipo} necesita lotes de entrenamiento para inicializarse")
        with contexto_autocast(opt):
            initialize_lora_eva_weights(modelo, dataloader=lotes_init, show_progress_bar=False)
    if var.muta_base and dir_init is not None:
        modelo.save_pretrained(dir_init)  # adaptador inicial: permite convertir a LoRA estándar al guardar
    asegurar_entrenables_fp32(modelo)
    entrenables, totales = contar_parametros(modelo)
    info = {"tipo": cfg.lora_tipo, "descripcion": var.descripcion, "modulos": sorted(config_peft.target_modules),
            "entrenables": entrenables, "totales": totales, "porcentaje": 100.0 * entrenables / totales}
    print(f"  {var.descripcion}")
    print(f"  módulos: {', '.join(info['modulos'])}")
    print(f"  parámetros entrenables: {entrenables:,} de {totales:,} ({info['porcentaje']:.2f} %)")
    return modelo, info


listar_variantes()

In [ ]:
# @title 9 · Evaluación
@torch.no_grad()
def predict(model, loader, opt: dict):
    """Pasa un split por el modelo y devuelve (logits, etiquetas, índices) como arrays de numpy.

    Pone el modelo en modo evaluación (sin dropout) y al terminar lo deja en el modo en que estaba. Usa no_grad y
    no inference_mode porque se llama a mitad del entrenamiento: un tensor creado en inference_mode no se puede
    usar después en el grafo de gradientes.
    """
    was_training = model.training
    model.eval()
    logits, labels, indices = [], [], []
    for batch in loader:
        batch = mover(batch, opt)
        with contexto_autocast(opt):
            output = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        logits.append(output.float().cpu())
        labels.append(batch["labels"].cpu())
        indices.append(batch["indices"].cpu())
    model.train(was_training)
    return torch.cat(logits).numpy(), torch.cat(labels).numpy(), torch.cat(indices).numpy()


def macro_f1(labels: np.ndarray, preds: np.ndarray) -> float:
    """Macro-F1 sobre las clases presentes en el gold (dev no tiene Evidence ni Originator)."""
    return float(f1_score(labels, preds, labels=sorted(set(labels.tolist())), average="macro", zero_division=0))


# Las 5 clases con ejemplos en test; dev tiene además Future Work y Further Reading
CLASES_MACRO_5 = ("Background", "Uses", "Comparison", "Extends", "Motivation")


def metricas_comparables(labels: np.ndarray, preds: np.ndarray) -> dict:
    """F1-macro-9, F1-macro-5 y F1 ponderado, definidos como en la rama SciBERT (02_scibert_ab) para comparar.

    - f1_macro_9: promedio sobre las 9 etiquetas; las que no tienen ejemplos en el split cuentan 0.
    - f1_macro_5: promedio sobre CLASES_MACRO_5, en cualquier split. En test es igual a macro_f1 y al
      f1_macro_present de SciBERT.
    - f1_weighted: promedio ponderado por el soporte de cada clase.
    """
    nueve, cinco = list(range(len(ETIQUETAS))), [ETIQUETAS.index(c) for c in CLASES_MACRO_5]
    return {"f1_macro_9": float(f1_score(labels, preds, labels=nueve, average="macro", zero_division=0)),
            "f1_macro_5": float(f1_score(labels, preds, labels=cinco, average="macro", zero_division=0)),
            "f1_weighted": float(f1_score(labels, preds, average="weighted", zero_division=0))}


def bootstrap_ci(labels: np.ndarray, preds: np.ndarray, n: int = 1000, seed: int = 0) -> list[float]:
    """Intervalo de confianza del 95 % del macro-F1 por bootstrap (remuestreo de instancias con reemplazo)."""
    rng = np.random.default_rng(seed)
    scores = [macro_f1(labels[idx], preds[idx]) for idx in (rng.integers(0, len(labels), len(labels)) for _ in range(n))]
    return [float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))]


def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, bins: int = 15) -> float:
    """ECE: diferencia media entre confianza y acierto, ponderada por intervalos de confianza."""
    confidence, correct = probs.max(1), (probs.argmax(1) == labels).astype(float)
    ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        in_bin = (confidence > lo) & (confidence <= hi)
        if in_bin.any():
            ece += in_bin.mean() * abs(correct[in_bin].mean() - confidence[in_bin].mean())
    return float(ece)


def compute_metrics(logits: np.ndarray, labels: np.ndarray, sources: list[str], n_bootstrap: int) -> tuple[dict, np.ndarray]:
    """Calcula todas las métricas de un split a partir de los logits y devuelve (métricas, probabilidades).

    Incluye macro-F1 (con intervalo bootstrap si n_bootstrap > 0), accuracy, ECE, P/R/F1 por clase, macro-F1 por
    fuente, matriz de confusión y cuántas predicciones hubo de cada clase.
    """
    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
    preds = probs.argmax(1)
    present = sorted(set(labels.tolist()))
    p, r, f, s = precision_recall_fscore_support(labels, preds, labels=present, zero_division=0)
    sources = np.array(sources)
    metrics = {
        "n": int(len(labels)), "macro_f1": macro_f1(labels, preds), **metricas_comparables(labels, preds),
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1_ic95": bootstrap_ci(labels, preds, n_bootstrap) if n_bootstrap else None,
        "ece": expected_calibration_error(probs, labels),
        "clases_evaluadas": [ETIQUETAS[k] for k in present],
        "por_clase": {ETIQUETAS[k]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "soporte": int(s[i])}
                      for i, k in enumerate(present)},
        "por_fuente": {src: {"n": int((sources == src).sum()),
                             "macro_f1": macro_f1(labels[sources == src], preds[sources == src]),
                             "accuracy": float((preds[sources == src] == labels[sources == src]).mean())}
                       for src in sorted(set(sources.tolist()))},
        "confusion": confusion_matrix(labels, preds, labels=list(range(len(ETIQUETAS)))).tolist(),
        "predicciones_por_clase": {ETIQUETAS[k]: int(v) for k, v in enumerate(np.bincount(preds, minlength=len(ETIQUETAS))) if v},
    }
    return metrics, probs


def evaluate(model, loader, items: list[dict], opt: dict, n_bootstrap: int):
    """Evaluación final de un split: devuelve (métricas, predicciones por instancia con sus probabilidades)."""
    t0 = time.perf_counter()
    logits, labels, indices = predict(model, loader, opt)
    seconds = time.perf_counter() - t0
    sources = [items[i]["fuente"] for i in indices]
    metrics, probs = compute_metrics(logits, labels, sources, n_bootstrap)
    metrics["segundos"] = seconds
    metrics["ejemplos_por_s"] = len(labels) / max(seconds, 1e-9)
    predictions = sorted(({"instance_id": items[i]["instance_id"], "fuente": items[i]["fuente"],
                           "etiqueta": ETIQUETAS[int(labels[j])], "prediccion": ETIQUETAS[int(probs[j].argmax())],
                           "probs": {e: round(float(probs[j][k]), 5) for k, e in enumerate(ETIQUETAS)}}
                          for j, i in enumerate(indices)), key=lambda d: d["instance_id"])
    return metrics, predictions


def validate(model, loader, opt: dict) -> dict:
    """Evaluación rápida de dev durante el entrenamiento: macro-F1 (criterio del early stopping), accuracy y pérdida."""
    logits, labels, _ = predict(model, loader, opt)
    preds = logits.argmax(1)
    loss = F.cross_entropy(torch.from_numpy(logits), torch.from_numpy(labels)).item()
    return {"macro_f1": macro_f1(labels, preds), **metricas_comparables(labels, preds),
            "accuracy": float((preds == labels).mean()), "perdida": loss}


def print_metrics(m: dict, name: str) -> None:
    """Imprime las métricas de un split: resumen, tabla por clase, métricas por fuente y predicciones por clase."""
    ci = f" (IC95 {m['macro_f1_ic95'][0]:.3f}–{m['macro_f1_ic95'][1]:.3f})" if m["macro_f1_ic95"] else ""
    print(f"  {name}: macro-F1 {m['macro_f1']:.4f}{ci} · accuracy {m['accuracy']:.4f} · ECE {m['ece']:.3f} · "
          f"n {m['n']} · {m['ejemplos_por_s']:.1f} ej/s")
    print(f"  F1-macro-9 {m['f1_macro_9']:.4f} · F1-macro-5 {m['f1_macro_5']:.4f} · F1 ponderado {m['f1_weighted']:.4f} "
          f"(el macro-F1 de arriba promedia las {len(m['clases_evaluadas'])} clases presentes)")
    print(f"  {'clase':<17}{'P':>7}{'R':>7}{'F1':>7}{'n':>6}")
    for label, v in m["por_clase"].items():
        print(f"  {label:<17}{v['precision']:>7.3f}{v['recall']:>7.3f}{v['f1']:>7.3f}{v['soporte']:>6}")
    for source, v in m["por_fuente"].items():
        print(f"  fuente {source}: macro-F1 {v['macro_f1']:.4f} · accuracy {v['accuracy']:.4f} · n {v['n']}")
    print(f"  predicciones por clase: {m['predicciones_por_clase']}")

In [ ]:
# @title 10 · Entrenamiento
def set_seed(seed: int) -> None:
    """Fija las semillas de random, numpy y torch (CPU y todas las GPUs)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def compute_class_weights(items: list[dict], scheme: str, opt: dict):
    """Pesos por clase para la pérdida: None (ninguno), inverso de la frecuencia o su raíz cuadrada."""
    if scheme == "ninguno":
        return None
    counts = np.bincount([x["etiqueta"] for x in items], minlength=len(ETIQUETAS)).astype(float)
    weights = np.where(counts > 0, counts.sum() / (len(ETIQUETAS) * np.maximum(counts, 1)), 0.0)
    if scheme == "raiz_inversa":
        weights = np.sqrt(weights)
    return torch.tensor(weights, dtype=torch.float32, device=opt["dispositivo"])


def compute_loss(logits, labels, class_weights=None):
    """Entropía cruzada en fp32, con pesos por clase opcionales."""
    return F.cross_entropy(logits.float(), labels, weight=class_weights)


def create_optimizer(model, variant: VarianteLoRA, cfg: Config, opt: dict):
    """Crea el optimizador: AdamW (fusionado en CUDA), o el propio de LoRA+ o LoRA-FA según la variante."""
    extra = {"fused": True} if opt["adamw_fusionado"] else {}
    if variant.optimizador == "lora_plus":
        return create_loraplus_optimizer(model=model, optimizer_cls=torch.optim.AdamW, lr=cfg.lr,
                                         loraplus_lr_ratio=16.0, weight_decay=cfg.weight_decay, **extra)
    if variant.optimizador == "lora_fa":
        return create_lorafa_optimizer(model=model, r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lr=cfg.lr,
                                       weight_decay=cfg.weight_decay)
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay, **extra)


def build_scheduler(name: str, optimizer: Optimizer, warmup_steps: int, total_steps: int) -> LRScheduler:
    """Crea el planificador de la tasa de aprendizaje `name`. Todos empiezan con `warmup_steps` de calentamiento lineal.

    - linear: decae en línea recta hasta 0 (BERT, RoBERTa con LoRA; Mosbach et al., 2021).
    - cosine: decae hasta 0 siguiendo medio coseno (Loshchilov y Hutter, 2017).
    - cosine_min: medio coseno, pero solo hasta el 10 % de la tasa (LLaMA; Touvron et al., 2023).
    - constant: se queda constante (QLoRA, Dettmers et al., 2023; Pareja et al., 2024). No depende de cuántos pasos se
      planearon, así que encaja con el early stopping.
    - wsd: constante y decae en el 20 % final con forma 1-sqrt (MiniCPM, Hu et al., 2024; Hägele et al., 2024).
    - plateau: constante; train() la baja a la mitad cada vez que el macro-F1 de dev no mejora.
    """
    if name == "cosine":
        return get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    if name == "cosine_min":
        return get_cosine_with_min_lr_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps,
                                                           num_training_steps=total_steps, min_lr_rate=0.1)
    if name in ("constant", "plateau"):
        return get_constant_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps)
    if name == "wsd":
        return get_wsd_schedule(optimizer, num_warmup_steps=warmup_steps, num_decay_steps=max(1, int(0.2 * total_steps)),
                                num_training_steps=total_steps, decay_type="1-sqrt")
    return get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)


def create_scheduler(optimizer, num_training_steps: int, cfg: Config):
    """Planificador elegido en SCHEDULER, con calentamiento durante la fracción `warmup` de los pasos."""
    return build_scheduler(cfg.scheduler, optimizer, max(1, int(cfg.warmup * num_training_steps)), num_training_steps)


def compute_steps_per_epoch(train_loader, cfg: Config) -> int:
    """Pasos de optimizador por época: lotes por época divididos por la acumulación, redondeando hacia arriba."""
    return math.ceil(len(train_loader) / cfg.acumulacion)


def compute_num_training_steps(train_loader, cfg: Config) -> int:
    """Pasos de optimizador totales: pasos por época × épocas, acotado por max_pasos si está definido."""
    steps = compute_steps_per_epoch(train_loader, cfg) * cfg.epocas
    return min(steps, cfg.max_pasos) if cfg.max_pasos else steps


def compute_eval_steps(steps_per_epoch: int, num_training_steps: int, evals_per_epoch: int) -> set[int]:
    """Pasos de optimizador en los que se evalúa dev: `evals_per_epoch` puntos repartidos en cada época, más el último paso."""
    steps = {num_training_steps}
    epoch = 0
    while epoch * steps_per_epoch < num_training_steps:
        for k in range(1, evals_per_epoch + 1):
            step = epoch * steps_per_epoch + max(1, round(k * steps_per_epoch / evals_per_epoch))
            if step <= num_training_steps:
                steps.add(step)
        epoch += 1
    return steps


@dataclass
class EarlyStopping:
    """Sigue el macro-F1 de dev y decide cuándo parar.

    Una evaluación cuenta como mejora si supera al mejor valor en más de `min_delta`. Tras `patience` evaluaciones
    seguidas sin mejora, `should_stop` pasa a True. Con `patience=0` nunca para, pero sigue registrando el mejor paso.
    """

    patience: int
    min_delta: float = 0.0
    best_score: float = -math.inf
    best_step: int = 0
    num_bad_evals: int = 0

    def step(self, score: float, step: int) -> bool:
        """Registra el macro-F1 de una evaluación hecha en el paso `step`. Devuelve True si es el mejor hasta ahora."""
        if score > self.best_score + self.min_delta:
            self.best_score, self.best_step, self.num_bad_evals = score, step, 0
            return True
        self.num_bad_evals += 1
        return False

    @property
    def should_stop(self) -> bool:
        """True cuando se agotó la paciencia."""
        return self.patience > 0 and self.num_bad_evals >= self.patience


def get_trainable_state(model) -> dict:
    """Copia en CPU de los parámetros entrenables (adaptador y cabeza): basta para volver a un punto del entrenamiento."""
    return {name: p.detach().to("cpu", copy=True) for name, p in model.named_parameters() if p.requires_grad}


def load_trainable_state(model, state: dict) -> None:
    """Restaura en el modelo, en su sitio, los parámetros guardados con get_trainable_state."""
    params = dict(model.named_parameters())
    with torch.no_grad():
        for name, value in state.items():
            params[name].copy_(value.to(params[name].device))


def training_step(forward_model, batch: dict, cfg: Config, opt: dict, class_weights, scaler):
    """Forward y backward de un lote. Divide la pérdida por la acumulación y la escala en fp16. Devuelve la pérdida."""
    with contexto_autocast(opt):
        logits = forward_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
    loss = compute_loss(logits, batch["labels"], class_weights) / cfg.acumulacion
    (scaler.scale(loss) if scaler is not None else loss).backward()
    return loss.detach()


def optimizer_step(params, optimizer, scheduler, scaler, cfg: Config):
    """Recorta la norma del gradiente, actualiza los pesos y la tasa y borra los gradientes. Devuelve la norma."""
    if scaler is not None:
        scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(params, cfg.max_grad_norm) if cfg.max_grad_norm > 0 else None
    if scaler is not None:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    return grad_norm


def is_compile_error(error: BaseException) -> bool:
    """True si el error, o alguna de sus causas encadenadas, viene de torch.compile (dynamo, inductor o triton)."""
    chain, e = [], error
    while e is not None and len(chain) < 10:
        chain.append(f"{type(e).__module__}.{type(e).__name__}")
        e = e.__cause__ or e.__context__
    text = " ".join(chain)
    return any(k in text for k in ("torch._dynamo", "torch._inductor", "triton", "BackendCompilerFailed", "InductorError"))


def compile_model(model, batch: dict, cfg: Config, opt: dict, class_weights):
    """Compila el modelo y lo calienta con un lote real (forward + backward). Devuelve (modelo para el forward, info).

    Si la compilación no aplica o falla, devuelve el modelo sin compilar. Los gradientes del calentamiento se borran.
    """
    if not opt["compilar"]:
        return model, {"compilado": False, "motivo": opt["motivos"]["compilar"], "segundos": 0.0}
    t0 = time.perf_counter()
    try:
        # dynamic=True: un solo grafo para todas las longitudes y tamaños de lote. Sin él, dynamo recompila al ver
        # la segunda longitud y otra vez con el último lote (más corto), y la GPU queda parada minutos cada vez.
        compiled = torch.compile(model, dynamic=True,
                                 mode=None if cfg.modo_compilacion == "default" else cfg.modo_compilacion)
        training_step(compiled, batch, cfg, opt, class_weights, scaler=None)
        if opt["dispositivo"] == "cuda":
            torch.cuda.synchronize()
        info = {"compilado": True, "motivo": "ok", "segundos": time.perf_counter() - t0}
        print(f"  torch.compile listo en {info['segundos']:.1f} s")
        return compiled, info
    except Exception as e:  # noqa: BLE001
        torch._dynamo.reset()
        info = {"compilado": False, "motivo": f"falló: {type(e).__name__}: {str(e)[:300]}", "segundos": time.perf_counter() - t0}
        print(f"AVISO: torch.compile falló y sigo sin compilar · {info['motivo']}")
        return model, info
    finally:
        model.zero_grad(set_to_none=True)


def train(model, forward_model, train_loader, dev_loader, optimizer, scheduler, num_training_steps: int,
          cfg: Config, opt: dict, class_weights, compile_info: dict, log_callback=None) -> dict:
    """Bucle de entrenamiento con acumulación de gradiente, evaluación periódica de dev y early stopping.

    - `forward_model` es el modelo compilado (o el mismo `model` si no se compiló). Si la compilación falla a
      mitad, sigue con `model`.
    - Evalúa dev en los pasos de compute_eval_steps con el modelo sin compilar, para no forzar recompilaciones.
      Cada vez que el macro-F1 mejora, guarda en CPU una copia de los pesos entrenables.
    - Para cuando el macro-F1 lleva `cfg.paciencia` evaluaciones sin mejorar. Al terminar, por early stopping
      o por agotar los pasos, restaura en `model` los pesos del mejor paso.
    - Con SCHEDULER=plateau, cada evaluación sin mejora baja la tasa a la mitad antes de seguir.
    - Muestra una barra de progreso por paso de optimizador (BARRA_PROGRESO=0 la quita).
    - Si se pasa `log_callback(tipo, fila)`, lo llama con cada fila del registro ("train") y con cada evaluación de
      dev ("dev"). Así MLflow recibe las curvas mientras entrena.
    - Devuelve tiempos (sin contar las evaluaciones), memoria pico, el registro por paso y el de evaluaciones.
    """
    device = opt["dispositivo"]
    params = [p for p in model.parameters() if p.requires_grad]
    scaler = torch.amp.GradScaler(device) if opt["escalador"] else None
    eval_steps = compute_eval_steps(compute_steps_per_epoch(train_loader, cfg), num_training_steps, cfg.evals_por_epoca)
    early_stopping = EarlyStopping(patience=cfg.paciencia, min_delta=cfg.delta_minimo)
    best_state, step_log, eval_log = None, [], []
    step, examples, tokens, stopped_early = 0, 0, 0, False
    window_loss, window_batches, window_examples = torch.zeros((), device=device), 0, 0
    eval_seconds = window_eval_seconds = 0.0
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = t_window = time.perf_counter()
    model.train()
    # Avanza un paso por actualización de pesos. Va por stdout, como los mensajes del bucle (progress.write), para
    # que el kernel no los entregue desordenados; si fuera por stderr, la CLI mezclaría la barra con los mensajes.
    progress = tqdm(total=num_training_steps, desc="entrenando", unit="paso", ncols=100, mininterval=2,
                    file=sys.stdout, disable=not cfg.barra_progreso)
    for epoch in range(1, cfg.epocas + 1):
        for i, batch in enumerate(train_loader):
            batch = mover(batch, opt)
            try:
                loss = training_step(forward_model, batch, cfg, opt, class_weights, scaler)
            except Exception as e:  # noqa: BLE001
                if forward_model is model or not is_compile_error(e):
                    raise
                progress.write(f"AVISO: torch.compile falló en el paso {step + 1} ({type(e).__name__}); sigo sin compilar.")
                compile_info.update(compilado=False, motivo=f"falló en el paso {step + 1}: {type(e).__name__}")
                forward_model = model
                optimizer.zero_grad(set_to_none=True)
                loss = training_step(forward_model, batch, cfg, opt, class_weights, scaler)
            window_loss += loss
            window_batches += 1
            n = batch["labels"].shape[0]
            examples, window_examples = examples + n, window_examples + n
            tokens += int(batch["attention_mask"].sum())
            if (i + 1) % cfg.acumulacion and (i + 1) != len(train_loader):
                continue
            grad_norm = optimizer_step(params, optimizer, scheduler, scaler, cfg)
            step += 1
            progress.update(1)

            if step % cfg.log_cada == 0 or step == num_training_steps:
                now = time.perf_counter()
                row = {"paso": step, "epoca": epoch, "perdida": float(window_loss) * cfg.acumulacion / window_batches,
                       "lr": scheduler.get_last_lr()[0], "norma_grad": float(grad_norm) if grad_norm is not None else None,
                       "ejemplos_por_s": window_examples / max(now - t_window - window_eval_seconds, 1e-9)}
                step_log.append(row)
                if log_callback:
                    log_callback("train", row)
                progress.set_postfix(perdida=f"{row['perdida']:.4f}", refresh=False)
                progress.write(f"  paso {step:>5}/{num_training_steps} · época {epoch} · pérdida {row['perdida']:.4f} · "
                               f"lr {row['lr']:.2e} · {row['ejemplos_por_s']:.1f} ej/s")
                window_loss.zero_()
                window_batches, window_examples, window_eval_seconds, t_window = 0, 0, 0.0, now

            if step in eval_steps:
                t_eval = time.perf_counter()
                result = validate(model, dev_loader, opt)
                seconds = time.perf_counter() - t_eval
                eval_seconds += seconds
                window_eval_seconds += seconds
                improved = early_stopping.step(result["macro_f1"], step)
                if improved:
                    best_state = get_trainable_state(model)
                eval_log.append({"paso": step, "epoca": epoch, **result, "mejor": improved, "segundos": seconds,
                                 "lr": scheduler.get_last_lr()[0]})
                if log_callback:
                    log_callback("dev", eval_log[-1])
                state = ("mejor hasta ahora" if improved
                         else f"sin mejora {early_stopping.num_bad_evals}/{cfg.paciencia or '∞'}")
                if cfg.scheduler == "plateau" and not improved and not early_stopping.should_stop:
                    # LambdaLR multiplica base_lrs por el factor del paso: bajarlas reduce la tasa desde el paso siguiente.
                    scheduler.base_lrs = [lr * 0.5 for lr in scheduler.base_lrs]
                    state += f" · la tasa baja a {scheduler.base_lrs[0]:.2e}"
                progress.write(f"  dev en el paso {step}: macro-F1 {result['macro_f1']:.4f} · accuracy {result['accuracy']:.4f} · "
                               f"pérdida {result['perdida']:.4f} · {state}")
                if early_stopping.should_stop:
                    stopped_early = True
                    progress.write(f"  early stopping: el macro-F1 de dev no mejora desde el paso {early_stopping.best_step}")
                    break
            if step >= num_training_steps:
                break
        if stopped_early or step >= num_training_steps:
            break
    progress.close()

    if device == "cuda":
        torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    restored = best_state is not None and early_stopping.best_step != step
    if restored:
        load_trainable_state(model, best_state)
        print(f"  restauro los pesos del paso {early_stopping.best_step} (macro-F1 dev {early_stopping.best_score:.4f}); "
              f"el último paso fue el {step}")
    train_seconds = max(seconds - eval_seconds, 1e-9)
    memory = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else None
    return {"pasos": step, "pasos_planeados": num_training_steps, "ejemplos": examples, "segundos": seconds,
            "segundos_eval_dev": eval_seconds, "ejemplos_por_s": examples / train_seconds,
            "tokens_por_s": tokens / train_seconds, "memoria_pico_gb": memory,
            "mejor_paso": early_stopping.best_step, "mejor_macro_f1_dev": early_stopping.best_score,
            "parada_temprana": stopped_early, "pesos_restaurados": restored,
            "compilado_al_final": forward_model is not model, "registro": step_log, "evaluaciones": eval_log}

In [ ]:
# @title 11 · Guardado y MLflow
def _a_json(obj):
    """Convierte a algo serializable los objetos que json no sabe escribir (dtype, Path, números de numpy)."""
    if isinstance(obj, (torch.dtype, Path)):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    return str(obj)


def _reponer_modulos_guardados(modelo, carpeta_adaptador: Path) -> None:
    """La conversión PiSSA/OLoRA → LoRA de PEFT 0.21 omite los modules_to_save (la cabeza entrenada).

    Sin ellos el adaptador no se puede recargar; aquí se vuelven a escribir en el mismo archivo.
    """
    archivo = carpeta_adaptador / "adapter_model.safetensors"
    convertido = load_file(archivo)
    faltan = {k: v.detach().cpu().contiguous() for k, v in get_peft_model_state_dict(modelo).items()
              if k not in convertido and ".lora_" not in k}
    if faltan:
        convertido.update(faltan)
        save_file(convertido, archivo, metadata={"format": "pt"})


def guardar_corrida(cfg: Config, modelo, tok, var: VarianteLoRA, dir_init: Path | None, archivos: dict) -> Path:
    """Guarda adaptador, tokenizador, JSONs y predicciones en una carpeta por corrida y la empaqueta en un zip.

    PiSSA y OLoRA se guardan convertidos a LoRA estándar, para cargarlos sobre la base original.
    """
    base = Path(cfg.salidas_dir)
    carpeta = base / cfg.nombre_corrida
    carpeta.mkdir(parents=True, exist_ok=True)
    conversion = str(dir_init) if (var.muta_base and dir_init is not None) else None
    modelo.save_pretrained(carpeta / "adaptador", path_initial_model_for_weight_conversion=conversion)
    if conversion:
        _reponer_modulos_guardados(modelo, carpeta / "adaptador")
    tok.save_pretrained(carpeta / "adaptador")
    for nombre, contenido in archivos.items():
        with open(carpeta / nombre, "w", encoding="utf-8") as f:
            if nombre.endswith(".jsonl"):
                f.writelines(json.dumps(fila, ensure_ascii=False, default=_a_json) + "\n" for fila in contenido)
            else:
                json.dump(contenido, f, ensure_ascii=False, indent=2, default=_a_json)
    zip_corrida = Path(shutil.make_archive(str(base / cfg.nombre_corrida), "zip", root_dir=carpeta))
    shutil.copyfile(zip_corrida, base / "ultima_corrida.zip")
    return zip_corrida


class RegistroMLflow:
    """Registra una corrida en un servidor de MLflow desde un hilo aparte, sin frenar ni tumbar el entrenamiento.

    - Lo que se registra (parámetros, métricas con su paso, artefactos y el cierre) entra a una cola, y un hilo lo
      envía en orden. Si el servidor no responde, el hilo guarda lo pendiente y reintenta cada `reintento_s`
      segundos. Cuando vuelve, envía todo con el paso y la hora originales y sigue en la misma corrida, que termina
      en FINISHED: MLflow solo marca FAILED si el cliente se lo pide.
    - En cada intento prueba la IP de la configuración y, si existe, la escrita en `archivo_ip`. Así se puede
      cambiar la IP a mitad de corrida (en AWS Academy cambia al reiniciar la sesión, salvo con una Elastic IP).
    - La corrida se crea la primera vez que el servidor responde. Si al volver ya no la tiene (perdió su base de
      datos), crea otra con el mismo nombre y reenvía todo.
    - Solo descarta un envío si MLflow dice que su contenido es inválido (INVALID_PARAMETER_VALUE), o un artefacto
      que no se pudo subir con el servidor funcionando. Cualquier otro error, como un 403 porque la IP no está en el
      --allowed-hosts del servidor, se trata como caída y se reintenta.
    - cerrar() espera a que se envíe lo pendiente. Con el servidor caído espera como mucho hasta `espera_min`
      minutos después de que dejó de responder. Si no vuelve, cierra sin MLflow y la corrida queda en RUNNING e
      incompleta en el servidor. Los resultados locales no dependen de nada de esto.
    """

    CAMPOS_FILA = {"train": ("perdida", "lr", "norma_grad", "ejemplos_por_s", "epoca"),
                   "dev": ("macro_f1", "f1_macro_9", "f1_macro_5", "f1_weighted", "accuracy", "perdida")}

    def __init__(self, ip: str, experimento: str, nombre: str, espera_min: float, archivo_ip: Path,
                 reintento_s: float = 30.0):
        """Guarda la configuración y arranca el hilo que envía a MLflow."""
        self.ip, self.experimento, self.nombre, self.archivo_ip = ip, experimento, nombre, Path(archivo_ip)
        self.espera_s, self.reintento_s = espera_min * 60, reintento_s
        self.inicio_ms = int(time.time() * 1000)
        self.uri = self.cliente = self.run_id = self.experimento_id = None
        self.cola = queue.Queue()
        self.pendientes, self.enviados = [], []    # lo que falta enviar y lo ya enviado (por si hay que reenviar)
        self.caido_desde = None                    # time.monotonic() del primer intento fallido de la caída actual
        self.proximo_intento, self.descartados, self.recreaciones, self.ultimo_error = 0.0, 0, 0, None
        self.abandonado = self.avise_espera = False
        self.hilo = threading.Thread(target=self._bucle, name="mlflow", daemon=True)
        self.hilo.start()

    # Lo que llama el entrenamiento: solo encola, no espera a la red.
    def parametros(self, valores: dict) -> None:
        """Encola parámetros como texto (hasta 6000 caracteres), en tandas de 100 como pide MLflow."""
        items = [Param(k, str(v)[:6000]) for k, v in valores.items()]
        for i in range(0, len(items), 100):
            self.cola.put(("datos", {"params": items[i:i + 100]}))

    def etiquetas(self, valores: dict) -> None:
        """Encola etiquetas (texto); salta las que valen None."""
        items = [RunTag(k, str(v)[:5000]) for k, v in valores.items() if v is not None]
        for i in range(0, len(items), 100):
            self.cola.put(("datos", {"tags": items[i:i + 100]}))

    def metricas(self, valores: dict, paso: int = 0) -> None:
        """Encola métricas con su paso y la hora de ahora. Salta lo que no es un número (None, texto)."""
        ahora = int(time.time() * 1000)
        items = [Metric(k, float(v), ahora, int(paso)) for k, v in valores.items()
                 if isinstance(v, (int, float, np.integer, np.floating))]
        for i in range(0, len(items), 1000):
            self.cola.put(("datos", {"metrics": items[i:i + 1000]}))

    def registrar_fila(self, tipo: str, fila: dict) -> None:
        """Pasa a MLflow una fila del registro de entrenamiento ("train") o de una evaluación de dev ("dev")."""
        self.metricas({f"{tipo}/{k}": fila.get(k) for k in self.CAMPOS_FILA[tipo]}, fila["paso"])

    def artefactos(self, carpeta: Path, excluir: tuple = ()) -> None:
        """Encola cada archivo de `carpeta` como artefacto, con su subcarpeta. Salta las subcarpetas de `excluir`."""
        carpeta = Path(carpeta)
        for archivo in sorted(p for p in carpeta.rglob("*") if p.is_file()):
            relativo = archivo.relative_to(carpeta)
            if relativo.parts[0] not in excluir:
                destino = None if relativo.parent == Path(".") else relativo.parent.as_posix()
                self.cola.put(("artefacto", (str(archivo), destino)))

    def cerrar(self, estado: str = "FINISHED", esperar: bool = True) -> bool:
        """Encola el cierre con `estado` y espera al hilo. Devuelve True si todo llegó a MLflow.

        Con esperar=False (corrida que falló o se interrumpió) no espera a un servidor caído.
        """
        self.cola.put(("fin", (estado, esperar)))
        self.hilo.join()
        return not self.abandonado and self.run_id is not None

    def url_corrida(self) -> str | None:
        """Dirección de la corrida en la interfaz de MLflow, o None si todavía no se creó."""
        if self.run_id is None:
            return None
        return f"{self.uri}/#/experiments/{self.experimento_id}/runs/{self.run_id}"

    # Lo que hace el hilo.
    @staticmethod
    def _avisar(texto: str) -> None:
        """Imprime un aviso sin romper la barra de progreso del entrenamiento."""
        tqdm.write(f"  [mlflow] {texto}", file=sys.stdout)

    @staticmethod
    def _responde(uri: str) -> bool:
        """True si el endpoint /health del servidor contesta en menos de 5 s."""
        try:
            with urllib.request.urlopen(f"{uri}/health", timeout=5) as respuesta:
                return respuesta.status == 200
        except Exception:  # noqa: BLE001  (conexión rechazada, tiempo agotado, DNS, HTTP 5xx...)
            return False

    def _direcciones(self) -> list[str]:
        """URIs a probar: la IP de la configuración y, si existe y es distinta, la de `archivo_ip`."""
        candidatas = [self.ip]
        try:
            candidatas.append(self.archivo_ip.read_text(encoding="utf-8").strip())
        except OSError:
            pass
        uris = [c if c.startswith(("http://", "https://")) else f"http://{c}" for c in candidatas if c]
        return list(dict.fromkeys(uris))

    def _conectar(self) -> bool:
        """Busca una dirección que responda y acepte pedidos, y crea la corrida si hace falta. True si quedó lista.

        Con la conexión normal solo consulta /health. Al conectar por primera vez o al volver de una caída, además
        comprueba con un pedido real que la dirección acepta pedidos (un servidor puede responder /health y rechazar
        el resto) y que todavía tiene la corrida.
        """
        verificar = self.caido_desde is not None or self.run_id is None
        alguna_responde = False
        for uri in self._direcciones():
            if not self._responde(uri):
                continue
            alguna_responde = True
            if uri == self.uri and not verificar:
                return True
            cliente = self.cliente if uri == self.uri else MlflowClient(tracking_uri=uri)
            try:
                if self.run_id is not None:
                    try:
                        cliente.get_run(self.run_id)
                    except Exception as e:  # noqa: BLE001
                        if getattr(e, "error_code", "") != "RESOURCE_DOES_NOT_EXIST" or self.recreaciones >= 3:
                            raise
                        self._olvidar_corrida()
                if self.run_id is None:
                    experimento = cliente.get_experiment_by_name(self.experimento)
                    if experimento is None:
                        experimento_id = cliente.create_experiment(self.experimento)
                    else:
                        experimento_id = experimento.experiment_id
                        if experimento.lifecycle_stage == "deleted":
                            cliente.restore_experiment(experimento_id)
                    corrida = cliente.create_run(experimento_id, start_time=self.inicio_ms, run_name=self.nombre)
                    self.experimento_id, self.run_id = experimento_id, corrida.info.run_id
                    self._avisar(f"corrida «{self.nombre}»: {uri}/#/experiments/{experimento_id}/runs/{self.run_id}")
            except Exception as e:  # noqa: BLE001  (esta dirección no sirve: se prueba la siguiente)
                self.ultimo_error = f"{getattr(e, 'error_code', '') or type(e).__name__}: {str(e)[:200]}"
                continue
            if uri != self.uri:
                if self.uri is not None:
                    self._avisar(f"sigo en la nueva dirección {uri}")
                self.uri, self.cliente = uri, cliente
            return True
        if not alguna_responde:
            self.ultimo_error = None
        return False

    def _acepta_pedidos(self) -> bool:
        """True si el servidor actual responde /health y además contesta un pedido sobre la corrida."""
        if not self._responde(self.uri):
            return False
        try:
            self.cliente.get_run(self.run_id)
            return True
        except Exception:  # noqa: BLE001
            return False

    def _olvidar_corrida(self) -> None:
        """El servidor ya no tiene la corrida (perdió su base de datos): se creará otra y se reenviará todo."""
        self.recreaciones += 1
        self._avisar("el servidor ya no tiene la corrida (¿perdió su base de datos?): creo otra con el mismo nombre "
                     "y reenvío todo")
        self.run_id = None
        self.pendientes[:0], self.enviados = self.enviados, []

    def _tomar_lote(self, maximo: int) -> tuple[dict, int]:
        """Junta datos consecutivos del frente de la cola sin pasar los límites de log_batch (1000 en total)."""
        lote, n = {"metrics": [], "params": [], "tags": []}, 0
        for tipo, contenido in self.pendientes[:maximo]:
            if tipo != "datos":
                break
            (clave, items), = contenido.items()
            limite = 1000 if clave == "metrics" else 100
            if n and (len(lote[clave]) + len(items) > limite or sum(map(len, lote.values())) + len(items) > 1000):
                break
            lote[clave] += items
            n += 1
        return lote, n

    def _enviar(self) -> bool:
        """Envía lo pendiente en orden. Devuelve True cuando envió el cierre."""
        if not self._conectar():
            self._marcar_caida()
            return False
        self._marcar_recuperado()
        maximo = len(self.pendientes)
        while self.pendientes:
            tipo, contenido = self.pendientes[0]
            n = 1
            try:
                if tipo == "fin":
                    self.cliente.set_terminated(self.run_id, status=contenido[0], end_time=int(time.time() * 1000))
                    self.pendientes.pop(0)
                    return True
                if tipo == "artefacto":
                    self.cliente.log_artifact(self.run_id, contenido[0], contenido[1])
                else:
                    lote, n = self._tomar_lote(maximo)
                    self.cliente.log_batch(self.run_id, metrics=lote["metrics"], params=lote["params"], tags=lote["tags"])
                    self.enviados += self.pendientes[:n]
                del self.pendientes[:n]
                maximo = len(self.pendientes)
            except Exception as e:  # noqa: BLE001
                codigo = getattr(e, "error_code", "")
                if codigo == "RESOURCE_DOES_NOT_EXIST" and self.recreaciones < 3:
                    self._olvidar_corrida()
                    return False
                if tipo == "artefacto" and codigo != "INVALID_PARAMETER_VALUE" and self._acepta_pedidos():
                    # El servidor funciona pero el archivo no se pudo subir (por ejemplo, guarda los artefactos en
                    # S3 sin pasarlos por él y aquí no hay credenciales): se salta ese archivo y se sigue.
                    self.descartados += 1
                    if self.descartados <= 3:
                        self._avisar(f"no pude subir {Path(contenido[0]).name} ({type(e).__name__}: {str(e)[:150]}); "
                                     "el servidor responde, así que salto ese archivo")
                    self.pendientes.pop(0)
                    continue
                if codigo != "INVALID_PARAMETER_VALUE":
                    # Conexión, servidor caído o que rechaza el pedido (PERMISSION_DENIED si la IP no está en su
                    # --allowed-hosts): se trata como caída y se reintenta, sin descartar nada. El motivo solo se
                    # anota si /health contesta; si no, el aviso dice que el servidor no responde.
                    rechaza = self._responde(self.uri)
                    self.ultimo_error = f"{codigo or type(e).__name__}: {str(e)[:200]}" if rechaza else None
                    self._marcar_caida()
                    return False
                if n > 1:
                    maximo = 1     # el lote tenía algo inválido: se reintenta de a uno para descartar solo eso
                    continue
                self.descartados += 1
                if self.descartados <= 3:
                    self._avisar(f"MLflow rechazó un envío ({tipo}) por inválido y lo descarto: {str(e)[:200]}")
                self.pendientes.pop(0)
        return False

    def _marcar_caida(self) -> None:
        """Anota el primer intento fallido de una caída y cuándo toca el próximo intento."""
        ahora = time.monotonic()
        self.proximo_intento = ahora + self.reintento_s
        if self.caido_desde is None:
            self.caido_desde = ahora
            hasta = datetime.now() + timedelta(seconds=self.espera_s)
            motivo = f"rechaza los pedidos ({self.ultimo_error})" if self.ultimo_error else "no responde"
            if self.ultimo_error and "Invalid Host header" in self.ultimo_error:
                motivo += "; la dirección tiene que estar en el --allowed-hosts del servidor"
            self._avisar(f"el servidor {motivo} ({', '.join(self._direcciones())}). Sigo entrenando y reintento "
                         f"cada {self.reintento_s:.0f} s; al terminar lo espero hasta las {hasta:%H:%M}. "
                         f"Si cambió la IP, escríbela en {self.archivo_ip}.")

    def _marcar_recuperado(self) -> None:
        """Si venía de una caída, avisa que el servidor volvió."""
        if self.caido_desde is not None:
            self._avisar(f"el servidor volvió tras {(time.monotonic() - self.caido_desde) / 60:.1f} min; "
                         "envío lo pendiente")
            self.caido_desde, self.ultimo_error = None, None

    def _bucle(self) -> None:
        """Hilo: toma lo encolado, lo envía y reintenta mientras el servidor no responda, hasta procesar el cierre."""
        try:
            while True:
                if self.caido_desde is not None:
                    espera = max(0.05, self.proximo_intento - time.monotonic())
                else:
                    espera = 0.0 if self.pendientes else 5.0
                try:
                    self.pendientes.append(self.cola.get(timeout=espera) if espera else self.cola.get_nowait())
                    while True:
                        self.pendientes.append(self.cola.get_nowait())
                except queue.Empty:
                    pass
                fin = next((contenido for tipo, contenido in self.pendientes if tipo == "fin"), None)
                if self.abandonado:
                    self.pendientes.clear()
                    if fin:
                        return
                    continue
                if not self.pendientes or (self.caido_desde is not None and time.monotonic() < self.proximo_intento):
                    continue
                if self._enviar():
                    if self.descartados > 3:
                        self._avisar(f"en total se descartaron {self.descartados} envíos rechazados")
                    return
                if fin and self.caido_desde is not None:
                    limite = self.caido_desde + self.espera_s
                    if not fin[1] or time.monotonic() >= limite:
                        self.abandonado = True
                        self._avisar("el servidor no volvió a tiempo: cierro sin MLflow. Los resultados locales están "
                                     "completos; en MLflow la corrida quedó incompleta.")
                        return
                    if not self.avise_espera:
                        self.avise_espera = True
                        hasta = datetime.now() + timedelta(seconds=limite - time.monotonic())
                        self._avisar(f"terminó la corrida y el servidor no responde: lo espero hasta las {hasta:%H:%M}")
        except Exception as e:  # noqa: BLE001  (un error propio no debe tumbar el entrenamiento)
            self.abandonado = True
            self._avisar(f"error inesperado, dejo de enviar a MLflow: {type(e).__name__}: {e}")


def archivo_ip_mlflow() -> Path:
    """Archivo donde se puede escribir una IP nueva para MLflow: /content/mlflow_ip.txt en Colab, ./mlflow_ip.txt en local."""
    return Path("/content/mlflow_ip.txt") if en_colab() else Path("mlflow_ip.txt")


def abrir_mlflow(cfg: Config) -> RegistroMLflow | None:
    """Abre el registro en MLflow si MLFLOW_IP está definido y encola la Config como parámetros; si no, devuelve None.

    No espera al servidor: el entrenamiento empieza igual y el hilo avisa enseguida si no responde.
    """
    if not cfg.mlflow_ip:
        return None
    if MlflowClient is None:
        raise ImportError("MLFLOW_IP está definido pero mlflow no está instalado: pip install mlflow-skinny")
    titulo("MLflow")
    print(f"  servidor {cfg.mlflow_ip} · experimento {cfg.mlflow_experimento} · corrida {cfg.nombre_corrida}")
    registro = RegistroMLflow(cfg.mlflow_ip, cfg.mlflow_experimento, cfg.nombre_corrida, cfg.mlflow_espera_min,
                              archivo_ip_mlflow())
    # strategy, model, seed y selection_metric con los nombres de la rama SciBERT, para agrupar y comparar corridas
    registro.parametros(asdict(cfg) | {"strategy": cfg.configuracion, "model": cfg.modelo, "seed": cfg.semilla,
                                       "selection_metric": "dev_f1_macro_present"})
    registro.etiquetas({f"version.{k}": v for k, v in versiones().items()})
    return registro


def metricas_finales(salida: dict) -> dict:
    """Aplana para MLflow las métricas finales de cada split (global, por clase y por fuente) y el resumen.

    Dev va como dev_final/..., para no mezclarse con la curva dev/... del entrenamiento; test va como test/...
    Además van test_accuracy, test_f1_macro_9, test_f1_macro_present, test_f1_weighted, train_seconds, latency_ms y
    best_epoch, con los nombres y las definiciones de la rama SciBERT (02_scibert_ab), para compararlas en MLflow, y
    test_f1_macro_5, que en test vale lo mismo que test_f1_macro_present.
    """
    m = {}
    for split, r in salida["resultados"].items():
        x, p = r["metricas"], "dev_final" if split == "dev" else split
        m |= {f"{p}/{k}": x[k] for k in ("macro_f1", "f1_macro_9", "f1_macro_5", "f1_weighted", "accuracy", "ece", "n",
                                         "segundos", "ejemplos_por_s")}
        if x["macro_f1_ic95"]:
            m[f"{p}/macro_f1_ic95_inf"], m[f"{p}/macro_f1_ic95_sup"] = x["macro_f1_ic95"]
        for clase, v in x["por_clase"].items():
            m |= {f"{p}/{k}/{clase}": v[k] for k in ("f1", "precision", "recall", "soporte")}
        for fuente, v in x["por_fuente"].items():
            m |= {f"{p}/fuente/{fuente}/{k}": v[k] for k in ("macro_f1", "accuracy", "n")}
        m |= {f"{p}/predicciones/{clase}": n for clase, n in x["predicciones_por_clase"].items()}
    resumen = salida["resumen"]
    m |= {f"entrenamiento/{k}": v for k, v in resumen["entrenamiento"].items()}
    m |= {f"adaptador/{k}": resumen["lora"][k] for k in ("entrenables", "totales", "porcentaje")}
    m["compilacion/segundos"] = resumen["compilacion"]["segundos"]
    m["segundos_totales"] = resumen["segundos_totales"]
    if "test" in salida["resultados"]:
        t = salida["resultados"]["test"]["metricas"]
        m |= {"test_accuracy": t["accuracy"], "test_f1_macro_9": t["f1_macro_9"], "test_f1_macro_5": t["f1_macro_5"],
              "test_f1_macro_present": t["macro_f1"], "test_f1_weighted": t["f1_weighted"],
              "latency_ms": 1000 * t["segundos"] / t["n"]}
    m |= {"train_seconds": resumen["entrenamiento"]["segundos"], "best_epoch": resumen["seleccion"]["mejor_epoca"]}
    return m


def cerrar_mlflow(registro: RegistroMLflow, cfg: Config, salida: dict) -> None:
    """Envía las métricas finales, la GPU y los archivos de la corrida, y cierra en FINISHED.

    Los artefactos son los archivos de la carpeta de la corrida, los mismos del zip; el adaptador solo con
    MLFLOW_ADAPTADOR=1. Espera a que todo llegue (ver RegistroMLflow.cerrar) y dice dónde quedó.
    """
    hw = salida["resumen"]["hardware"]
    registro.etiquetas({"gpu": hw["gpu"] or hw["dispositivo"], "modelo": salida["resumen"]["modelo"]})
    registro.metricas(metricas_finales(salida))
    if salida["zip"]:
        registro.artefactos(Path(cfg.salidas_dir) / cfg.nombre_corrida, () if cfg.mlflow_adaptador else ("adaptador",))
    titulo("MLflow")
    print("  envío las métricas finales y los archivos de la corrida")
    completo = registro.cerrar("FINISHED")
    print(f"  {'listo' if completo else 'incompleto'}: {registro.url_corrida() or 'la corrida no llegó a crearse'}")

In [ ]:
# @title 12 · Orquestación
def liberar_memoria() -> None:
    """Libera la memoria de corridas anteriores: recolector de basura, caché de CUDA y gráficos compilados."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    with contextlib.suppress(Exception):
        torch._dynamo.reset()


def ejecutar_experimento(cfg: Config | None = None, devolver_modelo: bool = True, **cambios) -> dict:
    """Corre un experimento completo. Acepta una Config o cambios sueltos: ejecutar_experimento(lora_tipo="dora").

    Etapas: configuración, hardware, datos, modelo, tokenización, adaptador, entrenamiento con early stopping,
    evaluación con los mejores pesos y guardado. Devuelve la Config, el resumen, las métricas y predicciones por
    split, la ruta del zip y, si devolver_modelo=True, el modelo y el tokenizador.

    Para encadenar corridas con modelos grandes, borra antes el resultado anterior (`del resultado`) o usa
    devolver_modelo=False: si no, el modelo previo sigue ocupando memoria de la GPU.

    Con MLFLOW_IP también registra la corrida en MLflow mientras entrena (celda 11). Si el servidor falla, el
    entrenamiento sigue y los resultados locales se guardan igual. Si la corrida falla, queda en FAILED en MLflow.
    """
    cfg = cfg or construir_config(**cambios)
    registro = abrir_mlflow(cfg)
    try:
        salida = _ejecutar_experimento(cfg, devolver_modelo, registro)
    except BaseException as e:
        if registro is not None:
            registro.cerrar("KILLED" if isinstance(e, KeyboardInterrupt) else "FAILED", esperar=False)
        raise
    if registro is not None:
        cerrar_mlflow(registro, cfg, salida)
    return salida


def _ejecutar_experimento(cfg: Config, devolver_modelo: bool, registro: RegistroMLflow | None) -> dict:
    """Las etapas de ejecutar_experimento con la Config ya armada. `registro` (o None) recibe las curvas del entrenamiento."""
    liberar_memoria()
    t_inicio = time.perf_counter()
    titulo("Configuración")
    imprimir_config(cfg)
    set_seed(cfg.semilla)

    titulo("Hardware y optimizaciones")
    hw = detectar_hardware()
    opt = decidir_optimizaciones(hw, cfg)
    aplicar_optimizaciones_globales(opt)
    imprimir_decisiones(hw, opt)
    var, avisos = resolver_variante(cfg, opt)
    for a in avisos:
        print(f"AVISO: {a}")

    titulo("Datos")
    datos = cargar_datos(cfg)

    titulo("Modelo")
    textos_humo = [" ".join(partes_texto(f, cfg)) for f in datos["train"]] if cfg.modelo == "humo" else None
    modelo, tok, ruta_modelo = crear_modelo(cfg, opt, var, textos_humo)
    print(f"  base: {ruta_modelo}")

    titulo("Tokenización")
    splits = ["train", "dev"] + (["test"] if cfg.evaluar_test else [])
    items = {s: tokenizar_split(datos[s], tok, cfg) for s in splits}
    for s in splits:
        r = resumen_tokens(items[s])
        print(f"  {s:<6}tokens media {r['media']:.0f} · p95 {r['p95']:.0f} · máx {r['max']} · "
              f"truncados {r['truncados_pct']:.1f} % (máx {cfg.max_longitud})")
    dl_train = crear_dataloader(items["train"], cfg.batch, True, cfg, opt, tok.pad_token_id)
    dl_eval = {s: crear_dataloader(items[s], cfg.batch_eval, False, cfg, opt, tok.pad_token_id) for s in splits[1:]}

    titulo("Adaptador")
    lotes_init = None
    if var.necesita_datos:
        lotes_init = [{k: v for k, v in mover(l, opt).items() if k in ("input_ids", "attention_mask")}
                      for l in itertools.islice(dl_train, 8)]
    dir_init = Path(cfg.salidas_dir) / "_tmp" / cfg.nombre_corrida / "adaptador_inicial" if var.muta_base else None
    modelo, info_lora = aplicar_lora(modelo, cfg, var, opt, lotes_init, dir_init)

    titulo("Entrenamiento")
    class_weights = compute_class_weights(items["train"], cfg.pesos_clase, opt)
    num_training_steps = compute_num_training_steps(dl_train, cfg)
    optimizer = create_optimizer(modelo, var, cfg, opt)
    scheduler = create_scheduler(optimizer, num_training_steps, cfg)
    lote0 = mover(next(iter(dl_train)), opt)
    verificar_pooling(lote0, tok.pad_token_id)
    modelo_fw, info_compilacion = compile_model(modelo, lote0, cfg, opt, class_weights)
    print(f"  {num_training_steps} pasos · lote {cfg.batch} × acumulación {cfg.acumulacion} · {len(items['train'])} ejemplos · "
          f"dev {cfg.evals_por_epoca} veces por época · paciencia {cfg.paciencia or 'sin límite'}")
    entrenamiento = train(modelo, modelo_fw, dl_train, dl_eval["dev"], optimizer, scheduler, num_training_steps,
                          cfg, opt, class_weights, info_compilacion,
                          log_callback=registro.registrar_fila if registro is not None else None)
    print(f"  {entrenamiento['segundos']:.1f} s ({entrenamiento['segundos_eval_dev']:.1f} s evaluando dev) · "
          f"{entrenamiento['ejemplos_por_s']:.1f} ej/s · {entrenamiento['tokens_por_s']:.0f} tokens/s"
          + (f" · memoria pico {entrenamiento['memoria_pico_gb']:.2f} GB" if entrenamiento["memoria_pico_gb"] else ""))

    titulo("Evaluación (pesos del mejor paso)")
    n_bootstrap = 1000 if cfg.modo == "completo" else 200
    resultados = {}
    for s, dl in dl_eval.items():
        metricas, predicciones = evaluate(modelo, dl, items[s], opt, n_bootstrap)
        print_metrics(metricas, s)
        resultados[s] = {"metricas": metricas, "predicciones": predicciones}

    opt_json = {k: v for k, v in opt.items() if k != "dtype"} | {"dtype": str(opt["dtype"])}
    seleccion = {"criterio": "macro-F1 de dev", "evals_por_epoca": cfg.evals_por_epoca, "paciencia": cfg.paciencia,
                 "delta_minimo": cfg.delta_minimo, "mejor_paso": entrenamiento["mejor_paso"],
                 "mejor_macro_f1_dev": entrenamiento["mejor_macro_f1_dev"],
                 "mejor_epoca": round(entrenamiento["mejor_paso"] / compute_steps_per_epoch(dl_train, cfg), 2),
                 "parada_temprana": entrenamiento["parada_temprana"], "pasos_entrenados": entrenamiento["pasos"],
                 "pasos_planeados": entrenamiento["pasos_planeados"]}
    resumen = {
        "corrida": cfg.nombre_corrida, "modo": cfg.modo, "modelo": ruta_modelo if cfg.modelo != "humo" else "humo",
        "lora": info_lora, "configuracion": cfg.configuracion, "avisos": avisos,
        "metricas": {s: {k: r["metricas"][k] for k in ("macro_f1", "macro_f1_ic95", "f1_macro_9", "f1_macro_5",
                                                        "f1_weighted", "accuracy", "ece", "n")}
                     for s, r in resultados.items()},
        "seleccion": seleccion,
        "entrenamiento": {k: v for k, v in entrenamiento.items() if k not in ("registro", "evaluaciones")},
        "compilacion": info_compilacion, "precision": opt["precision"], "hardware": hw,
        "segundos_totales": time.perf_counter() - t_inicio,
    }
    zip_corrida = None
    if cfg.guardar:
        archivos = {"config.json": asdict(cfg), "resumen.json": resumen, "optimizaciones.json": opt_json,
                    "versiones.json": versiones(), "registro_entrenamiento.json": entrenamiento["registro"],
                    "evaluaciones_dev.json": entrenamiento["evaluaciones"]}
        for s, r in resultados.items():
            archivos[f"metricas_{s}.json"] = r["metricas"]
            archivos[f"predicciones_{s}.jsonl"] = r["predicciones"]
        zip_corrida = guardar_corrida(cfg, modelo, tok, var, dir_init, archivos)

    titulo("Resumen")
    dev = resumen["metricas"]["dev"]
    print(f"  {cfg.nombre_corrida}")
    print(f"  macro-F1 dev {dev['macro_f1']:.4f} · accuracy {dev['accuracy']:.4f} · "
          f"{info_lora['entrenables']:,} parámetros entrenables ({info_lora['porcentaje']:.2f} %)")
    for s, x in resumen["metricas"].items():
        print(f"  {s}: F1-macro-9 {x['f1_macro_9']:.4f} · F1-macro-5 {x['f1_macro_5']:.4f} · F1 ponderado {x['f1_weighted']:.4f}")
    print(f"  mejor paso {seleccion['mejor_paso']} (época {seleccion['mejor_epoca']}) de {seleccion['pasos_entrenados']} "
          "entrenados" + (" (early stopping)" if seleccion["parada_temprana"] else ""))
    print(f"  {opt['precision']} · compilado: {'sí' if info_compilacion['compilado'] else 'no'} · "
          f"{resumen['segundos_totales']:.0f} s en total")
    if zip_corrida:
        print(f"  resultados: {zip_corrida} (copia en {Path(cfg.salidas_dir) / 'ultima_corrida.zip'})")
    salida = {"cfg": cfg, "resumen": resumen, "resultados": resultados, "zip": zip_corrida}
    if devolver_modelo:
        salida |= {"modelo": modelo, "tokenizer": tok}
    return salida


def probar_variantes(variantes=None, **cambios) -> list[dict]:
    """Entrena 2 pasos en modo mini con cada variante, guarda y evalúa. Sirve para verificar el hardware local."""
    tabla = []
    base = Path(construir_config(modo="mini").salidas_dir) / "pruebas_variantes"
    for nombre in variantes or list(VARIANTES_LORA):
        t0 = time.perf_counter()
        try:
            r = ejecutar_experimento(modo="mini", lora_tipo=nombre, max_pasos=2, mini_train=32, mini_dev=16,
                                     salidas_dir=str(base), nombre_corrida=f"prueba-{nombre}",
                                     devolver_modelo=False, mlflow_ip="", **cambios)  # sin MLflow
            estado = f"ok · macro-F1 dev {r['resumen']['metricas']['dev']['macro_f1']:.3f}"
            if r["resumen"]["avisos"]:
                estado += " · " + "; ".join(r["resumen"]["avisos"])
            del r
        except Exception as e:  # noqa: BLE001
            estado = f"ERROR {type(e).__name__}: {str(e)[:200]}"
        tabla.append({"variante": nombre, "estado": estado, "segundos": round(time.perf_counter() - t0, 1)})
        liberar_memoria()
    titulo("Prueba de variantes")
    for fila in tabla:
        print(f"  {fila['variante']:<10}{fila['segundos']:>7.1f} s  {fila['estado']}")
    return tabla

## Ejecutar

La celda 13 corre un experimento con la configuración de la celda 2 y las variables de entorno. Con la CLI de Colab, el kernel conserva estas funciones entre llamadas, así que se pueden lanzar más corridas sin repetir el notebook:

```bash
echo 'r = ejecutar_experimento(lora_tipo="dora", configuracion="B")' | colab exec -s lora --timeout 14400
```

In [ ]:
# @title 13 · Ejecutar el experimento
resultado = ejecutar_experimento()

In [ ]:
# @title 14 · Probar todas las variantes (opcional, PROBAR_VARIANTES=1)
if os.environ.get("PROBAR_VARIANTES", "0") == "1":
    tabla_variantes = probar_variantes()
else:
    print("Omitido. Con PROBAR_VARIANTES=1 se entrena 2 pasos en modo mini con cada variante de LoRA.")